# Chai-1: Multi-Modal Foundation Model for Molecular Structure Prediction

## CSV Batch Processor for High-Throughput Predictions

Easy-to-use batch processing interface for [Chai-1](https://github.com/chaidiscovery/chai-lab), a multi-modal foundation model for molecular structure prediction. Chai-1 achieves accuracy comparable to AlphaFold3 with comprehensive support for proteins, DNA, RNA, and small molecule ligands.

### **Key Features:**
- **Batch Processing**: CSV-based workflow for processing hundreds of predictions efficiently
- **State-of-the-Art Accuracy**: Performance comparable to AlphaFold3 across diverse biomolecular systems
- **Comprehensive Support**: Proteins, DNA, RNA, and ligands (SMILES notation)
- **Extended FASTA Input**: Uses Chai-1's extended FASTA format with molecule type headers
- **Automatic MSA Generation**: Integrated ColabFold MSA server for protein sequences
- **Google Drive Integration**: Automatic upload of results to your Drive
- **Calibration-Based Parallel GPU Scheduling**: Optimal GPU utilization for batch jobs

### **Important: GPU Requirements**
- **Chai-1 requires bfloat16 support (compute capability >= 8.0)**
- **T4 GPUs (compute capability 7.5) are NOT compatible**
- **Use A100, L4, or V100S (with fp32 fallback) in Colab**
- The notebook will detect your GPU and warn/abort if incompatible

### **Workflow:**
1. **Prepare CSV**: Create input file with sequences and ligands
2. **Configure**: Set prediction parameters (recycling, diffusion steps, MSA)
3. **Run Batch**: Automated processing with progress tracking
4. **Download**: Results automatically uploaded to Google Drive

### **CSV Input Format:**
- **Required columns**: `jobname`, `seq1_name`, `seq1_type`, `seq1`
- **Optional columns**: `seq1_copies`, `seq1_mods`, `pocket_binder`, `pocket_contacts`, `covalent_bonds`
- **Supports**: Up to 10 sequences per job (seq1 through seq10)
- **Sequence types**: protein, dna, rna, ligand (CCD code or SMILES)

### **Ligand Support:**
- **CCD Codes**: ATP, GTP, NAD, FAD, HEM, and 20+ common molecules (auto-converted to SMILES)
- **SMILES Strings**: Any valid SMILES notation for custom ligands
- **Ions**: Mg2+, Zn2+, Ca2+, Fe2+/3+, and other metal ions (auto-converted to SMILES)
- **Note**: Post-translational modifications (PTMs) are not directly supported by Chai-1's FASTA format

### **Output Format:**
- Chai-1 outputs mmCIF format only
- Files: `pred.model_idx_0.cif`, `pred.model_idx_1.cif`, etc.
- Scores: `scores.model_idx_0.npz`, etc.

### **Repository:**
- [Chai-1 GitHub Repository](https://github.com/chaidiscovery/chai-lab)
- [Chai-1 Technical Report](https://www.biorxiv.org/content/10.1101/2024.10.10.615955)

### **Citations:**

**Chai-1:**

[Chai Discovery. Chai-1: Decoding the molecular interactions of life. *bioRxiv*, 2024](https://www.biorxiv.org/content/10.1101/2024.10.10.615955)

**If using automatic MSA generation:**

[Mirdita M, Schutze K, Moriwaki Y, et al. ColabFold: making protein folding accessible to all. *Nature Methods*, 2022](https://doi.org/10.1038/s41592-022-01488-1)

### **Known Limitations:**
- **Ligands (CCD codes)**: The notebook converts known CCD codes to SMILES using a built-in reference table (28 common ligands). CCD codes not in the reference table will fail. **Workaround**: Use direct SMILES strings in your CSV for uncommon ligands. Jobs without ligands (protein-only, protein-DNA, protein-RNA, protein-ion) are unaffected.
- **GPU requirement**: Chai-1 requires bfloat16 support (compute capability >= 8.0). T4 GPUs are NOT supported. Minimum: L4 (paid) or A100 (Pro).

---

### **Quick Start Guide:**

1. **Run Cell 1**: Kernel restart (if needed — skips if already installed)
2. **Run Cell 2**: Install Chai-1 (takes ~3-5 minutes)
3. **Run Cell 3**: Initialize CSV processor
4. **Run Cell 4**: Upload your CSV file and connect Google Drive
5. **Run Cell 5**: Configure prediction parameters
6. **Run Cell 6**: Execute batch predictions (results auto-uploaded to Drive)

**Example CSV structure:**
```csv
jobname,seq1_name,seq1_type,seq1,seq2_name,seq2_type,seq2
kinase_atp,kinaseA,protein,MSEQVENCE...,atp_ligand,ligand,ATP
protein_dna,proteinB,protein,MKLLVV...,dna_seq,dna,ATCGATCG
```

For ligands: use CCD codes (ATP, GTP, etc.) or SMILES strings directly in the `seq` column.

---

**Ready to start? Run Cell 1 below!**

In [ ]:
#@title Cell 1: Kernel Restart (if needed)
import subprocess
import sys
import os
import re

# Restart marker to handle clean install
restart_marker = "/content/.chai1_install_restart"
is_post_restart = os.path.exists(restart_marker)

def run_cmd(cmd, desc):
    """Execute command with output suppression unless error"""
    print(f"[{desc}]")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"FAILED: {result.stderr[:500]}")
        return False
    print("OK")
    return True

def get_cuda_version():
    """Detect CUDA version from nvidia-smi"""
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
        if result.returncode == 0:
            match = re.search(r'CUDA Version: (\d+\.\d+)', result.stdout)
            if match:
                version = match.group(1)
                major = int(version.split('.')[0])
                minor = int(version.split('.')[1])
                return major, minor, version
    except Exception as e:
        print(f"Could not detect CUDA version: {e}")
    return None, None, None

def get_gpu_compute_capability():
    """Get GPU compute capability"""
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
            capture_output=True, text=True
        )
        if result.returncode == 0 and result.stdout.strip():
            cc = result.stdout.strip()
            parts = cc.split('.')
            if len(parts) == 2:
                return float(cc), int(parts[0]), int(parts[1])
    except:
        pass
    # Fallback: try to detect from GPU name
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            name = result.stdout.strip().lower()
            # Known GPU compute capabilities
            if 't4' in name:
                return 7.5, 7, 5
            elif 'a100' in name:
                return 8.0, 8, 0
            elif 'l4' in name:
                return 8.9, 8, 9
            elif 'v100' in name:
                return 7.0, 7, 0
            elif 'a10' in name:
                return 8.6, 8, 6
            elif 'h100' in name:
                return 9.0, 9, 0
            elif 'b200' in name or 'b100' in name or 'gb200' in name:
                return 12.0, 12, 0
            elif 'rtx 5' in name:
                return 12.0, 12, 0
            elif 'rtx 6000' in name:
                return 12.0, 12, 0
    except:
        pass
    return None, None, None

if not is_post_restart:
    # ==============================================================
    # PRE-RESTART: PREFLIGHT CHECKS
    # ==============================================================
    print("=" * 60)
    print("PREFLIGHT CHECKS")
    print("=" * 60)

    # Check GPU
    cuda_major, cuda_minor, cuda_version = get_cuda_version()
    if cuda_major is None:
        print("No CUDA detected - cannot proceed")
        sys.exit(1)

    print(f"CUDA Version: {cuda_version}")
    print(f"   Driver CUDA: {cuda_major}.{cuda_minor}")

    # Check GPU type
    result = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                          capture_output=True, text=True)
    if result.returncode == 0:
        gpu_name = result.stdout.strip()
        print(f"GPU: {gpu_name}")

    # CRITICAL: Check compute capability for bfloat16 support
    cc, cc_major, cc_minor = get_gpu_compute_capability()
    if cc is not None:
        print(f"Compute Capability: {cc}")
        if cc < 8.0:
            print()
            print("!" * 60)
            print("CRITICAL: GPU INCOMPATIBLE WITH CHAI-1")
            print("!" * 60)
            print(f"Your GPU has compute capability {cc}")
            print("Chai-1 REQUIRES bfloat16 support (compute capability >= 8.0)")
            print()
            print("Incompatible GPUs: T4 (7.5), V100 (7.0), P100 (6.0)")
            print("Compatible GPUs: A100 (8.0), L4 (8.9), A10G (8.6), H100 (9.0), Blackwell (12.0)")
            print()
            print("To fix: Go to Runtime -> Change runtime type -> Select A100 or L4 GPU")
            print("!" * 60)
            print()
            print("Installation will proceed but predictions WILL FAIL on this GPU.")
            print("You have been warned.")
            print()
    else:
        print("Could not detect compute capability - will check again before prediction")

    print()
    print("=" * 60)
    print("RESTARTING RUNTIME FOR CLEAN INSTALL")
    print("=" * 60)
    print("Runtime will restart in 2 seconds")
    print("After restart: Run this cell again to install Chai-1")
    print("=" * 60)

    with open(restart_marker, "w") as f:
        f.write("restarted")

    import time
    time.sleep(2)
    os.kill(os.getpid(), 9)



In [ ]:
#@title Cell 2: Upload CSV and Connect Google Drive
#@markdown Upload your CSV file and connect Google Drive. Then click **Run All Below** for hands-free execution.
%%time

import subprocess
import sys
import os

# Quick-install pydrive2 (not in Colab baseline)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydrive2"], capture_output=True, text=True)

from google.colab import files

# Configuration options
setup_google_drive = True #@param {type:"boolean"}
#@markdown - Setup Google Drive for automatic result upload

gdrive_folder_name = "Chai1_Results" #@param {type:"string"}
#@markdown - Google Drive folder name for batch results

msa_folder = ""  #@param {type:"string"}
#@markdown > **Pre-computed MSAs** (from MSA Colab). Leave empty for online MSA.
#@markdown > Enter a path, zip filename, or Google Drive zip name. The notebook will
#@markdown > resolve it to the `paired/` directory automatically.

print("=" * 60)
print("UPLOAD CSV AND CONNECT GOOGLE DRIVE")
print("=" * 60)

print("\nUpload your input CSV file with job specifications")
print("Required columns: jobname, seq1_name, seq1_type, seq1")
print("Optional: seq1_copies, seq1_mods (will warn if PTMs used)")
print("Supports up to 10 sequences per job (seq1 through seq10)")
print("Sequence types: protein, dna, rna, ligand")

uploaded_csv = files.upload()

if not uploaded_csv:
    raise ValueError("No CSV file uploaded")

csv_filename = list(uploaded_csv.keys())[0]
print(f"\nUploaded: {csv_filename}")

# Setup Google Drive
drive = None
if setup_google_drive:
    try:
        from pydrive2.drive import GoogleDrive
        from pydrive2.auth import GoogleAuth
        from google.colab import auth
        from oauth2client.client import GoogleCredentials

        print("\nSetting up Google Drive...")
        auth.authenticate_user()
        gauth = GoogleAuth()
        gauth.credentials = GoogleCredentials.get_application_default()
        drive = GoogleDrive(gauth)
        print("Google Drive connected")
    except Exception as e:
        print(f"Google Drive setup failed: {e}")
        drive = None

# ============================================================
# RESOLVE PRE-COMPUTED MSA FOLDER
# ============================================================
import os as _os

def resolve_msa_folder(msa_folder_input, _drive=None):
    """Resolve user's msa_folder input to the actual paired/ directory path."""
    if not msa_folder_input or not msa_folder_input.strip():
        return ''

    name = msa_folder_input.strip().rstrip('/')

    if name.startswith('/'):
        base_path = name
    else:
        base_path = f"/content/{name}"

    if _os.path.isdir(base_path):
        paired = _os.path.join(base_path, 'paired')
        if _os.path.isdir(paired) and _os.listdir(paired):
            print(f"   MSA folder resolved: {paired}")
            return paired
        if _os.path.basename(base_path) == 'paired' and _os.listdir(base_path):
            print(f"   MSA folder resolved: {base_path}")
            return base_path

    zip_path = f"{base_path}.zip"
    if not _os.path.isfile(zip_path):
        zip_path = f"/content/{_os.path.basename(name)}.zip"

    if _os.path.isfile(zip_path):
        print(f"   Found local zip: {zip_path}")
        import zipfile as _zf
        extract_to = _os.path.dirname(zip_path) or '/content'
        with _zf.ZipFile(zip_path, 'r') as zf:
            zf.extractall(extract_to)
        print(f"   Unzipped to {extract_to}")
        paired = _os.path.join(base_path, 'paired')
        if _os.path.isdir(paired) and _os.listdir(paired):
            print(f"   MSA folder resolved: {paired}")
            return paired

    if _drive:
        zip_filename = f"{_os.path.basename(name)}.zip"
        print(f"   Searching Google Drive for {zip_filename}...")
        try:
            file_list = _drive.ListFile({
                'q': f"title='{zip_filename}' and trashed=false"
            }).GetList()
            if file_list:
                gdrive_file = file_list[0]
                local_zip = f"/content/{zip_filename}"
                print(f"   Downloading {zip_filename} from Google Drive...")
                gdrive_file.GetContentFile(local_zip)
                print(f"   Downloaded ({_os.path.getsize(local_zip) / 1024 / 1024:.1f} MB)")

                import zipfile as _zf
                with _zf.ZipFile(local_zip, 'r') as zf:
                    zf.extractall('/content')
                print(f"   Unzipped to /content/")

                paired = _os.path.join(base_path, 'paired')
                if _os.path.isdir(paired) and _os.listdir(paired):
                    print(f"   MSA folder resolved: {paired}")
                    return paired
            else:
                print(f"   {zip_filename} not found on Google Drive")
        except Exception as e:
            print(f"   Google Drive download failed: {e}")

    print(f"   WARNING: Could not resolve MSA folder '{msa_folder_input}'")
    return ''

resolved_msa_folder = ''
if msa_folder:
    print(f"\nResolving MSA folder: {msa_folder}")
    resolved_msa_folder = resolve_msa_folder(msa_folder, drive)
    if resolved_msa_folder:
        print(f"   Using pre-computed MSAs from: {resolved_msa_folder}")
        msa_files = [f for f in _os.listdir(resolved_msa_folder) if f.endswith('_paired.a3m')]
        print(f"   Found {len(msa_files)} paired MSA files")

# Store settings (processor and jobs added in Cell 4)
global_settings = {
    'csv_filename': csv_filename,
    'drive': drive,
    'gdrive_folder_name': gdrive_folder_name,
    'msa_folder': resolved_msa_folder,
}

print("\n" + "=" * 60)
print(f"CSV uploaded, Google Drive {'connected' if drive else 'skipped'}.")
print("Run all cells below for hands-free install + predict.")
print("=" * 60)

In [ ]:
#@title Cell 3: Install Chai-1
#@markdown Installs Chai-1 with three-layer NumPy defense. Safe to re-run.
%%time

import subprocess
import sys
import os
import re

def run_cmd(cmd, desc):
    """Execute command with output suppression unless error"""
    print(f"   [{desc}]")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=600)
    if result.returncode != 0:
        print(f"   FAILED: {result.stderr[:500]}")
        return False
    print("   OK")
    return True

def get_gpu_compute_capability():
    """Get GPU compute capability"""
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
            capture_output=True, text=True
        )
        if result.returncode == 0 and result.stdout.strip():
            cc = result.stdout.strip()
            parts = cc.split('.')
            if len(parts) == 2:
                return float(cc), int(parts[0]), int(parts[1])
    except:
        pass
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            name = result.stdout.strip().lower()
            if 't4' in name: return 7.5, 7, 5
            elif 'a100' in name: return 8.0, 8, 0
            elif 'l4' in name: return 8.9, 8, 9
            elif 'v100' in name: return 7.0, 7, 0
            elif 'a10' in name: return 8.6, 8, 6
            elif 'h100' in name: return 9.0, 9, 0
            elif 'b200' in name or 'b100' in name or 'gb200' in name: return 12.0, 12, 0
            elif 'rtx 5' in name: return 12.0, 12, 0
            elif 'rtx 6000' in name: return 12.0, 12, 0
    except:
        pass
    return None, None, None

# ==============================================================
# SKIP IF ALREADY INSTALLED
# ==============================================================
ready_marker = "/content/CHAI1_READY"
restart_marker = "/content/.chai1_install_restart"

if os.path.exists(ready_marker):
    result = subprocess.run(
        [sys.executable, "-c",
         "import chai_lab; import numpy; "
         "assert numpy.__version__.startswith('1.2'); "
         "import torch; v = torch.__version__.split('+')[0].split('.'); "
         "assert int(v[0]) == 2 and int(v[1]) >= 7; "
         "print('OK')"],
        capture_output=True, text=True, timeout=30
    )
    if result.returncode == 0 and 'OK' in result.stdout:
        print("Chai-1 already installed and verified. Skipping.")
        if os.path.exists(restart_marker):
            os.remove(restart_marker)
        import time  # needed for %%time to work
        raise SystemExit(0)  # clean exit - won't kill kernel in Colab
    else:
        print("Previous install broken, reinstalling...")
        os.remove(ready_marker)

# ==============================================================
# THREE-LAYER DEFENSE: NumPy 1.x / Colab 2.x compatibility
# ==============================================================
print("=" * 60)
print("THREE-LAYER DEFENSE: NumPy compatibility")
print("=" * 60)

# Layer 2: sitecustomize.py -- force correct NumPy at interpreter startup
sitecustomize_content = (
    "# Colab NumPy priority fix for Chai-1\n"
    "import sys, os\n"
    "if '/env/python' in sys.path:\n"
    "    sys.path.remove('/env/python')\n"
    "if 'PYTHONPATH' in os.environ:\n"
    "    del os.environ['PYTHONPATH']\n"
)
sc_path = f"/usr/local/lib/python{sys.version_info.major}.{sys.version_info.minor}/dist-packages/sitecustomize.py"
with open(sc_path, "w") as f:
    f.write(sitecustomize_content)
print(f"   Layer 2: sitecustomize.py -> {sc_path}")

# IPython startup hook
ip_dir = "/root/.ipython/profile_default/startup"
os.makedirs(ip_dir, exist_ok=True)
with open(os.path.join(ip_dir, "00-fix_syspath.py"), "w") as f:
    f.write(sitecustomize_content)
print("   Layer 2b: IPython startup hook installed")

# Layer 3: Clean current kernel
if '/env/python' in sys.path:
    sys.path.remove('/env/python')
if 'PYTHONPATH' in os.environ:
    del os.environ['PYTHONPATH']
for mod in [k for k in list(sys.modules.keys()) if k.startswith(('numpy', 'pandas', 'scipy', 'numba'))]:
    del sys.modules[mod]
print("   Layer 3: sys.path cleaned, cached modules cleared")

# ==============================================================
# INSTALLATION
# ==============================================================
print("\n" + "=" * 60)
print("INSTALLING CHAI-1")
print("=" * 60)

# GPU info
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f"   GPU: {result.stdout.strip()}")

cc, cc_major, cc_minor = get_gpu_compute_capability()
if cc is not None:
    print(f"   Compute Capability: {cc}")
    if cc < 8.0:
        print()
        print("   " + "!" * 50)
        print("   WARNING: GPU does NOT support bfloat16 (cc < 8.0)")
        print("   Chai-1 predictions will FAIL on this GPU.")
        print("   Compatible: A100 (8.0), L4 (8.9), H100 (9.0), Blackwell (12.0)")
        print("   " + "!" * 50)

# ----------------------------------------------------------
# [1/6] NumPy 1.26.4 (Chai-1 requires numpy~=1.21 i.e. <2.0)
# ----------------------------------------------------------
print("\n" + "=" * 60)
print("[1/6] Installing NumPy 1.26.4 (Chai-1 requires <2.0)")
run_cmd(f"{sys.executable} -m pip install 'numpy==1.26.4'", "numpy==1.26.4")
result = subprocess.run(
    [sys.executable, "-c", "import numpy; print(numpy.__version__)"],
    capture_output=True, text=True
)
np_ver = result.stdout.strip()
print(f"   NumPy version: {np_ver}")
if not np_ver.startswith("1.26"):
    print("   ERROR: NumPy 1.26.4 not installed correctly!")
    sys.exit(1)

# ----------------------------------------------------------
# [2/6] PyTorch 2.7.1+cu128 (Blackwell sm_120 compatible)
#   Chai-1 pins torch<2.7 but devs confirmed 2.7 works (GitHub #381)
# ----------------------------------------------------------
print("\n" + "=" * 60)
print("[2/6] Installing PyTorch 2.7.1+cu128 (Blackwell sm_120 compatible)")
print("   Chai-1 pins torch<2.7 but devs confirmed 2.7+ works (GitHub #381)")
run_cmd(
    f"{sys.executable} -m pip uninstall -y torch torchvision torchaudio",
    "Removing Colab PyTorch 2.10.0"
)
run_cmd(
    f"{sys.executable} -m pip install torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 "
    f"--index-url https://download.pytorch.org/whl/cu128",
    "torch==2.7.1+cu128, torchvision==0.22.1, torchaudio==2.7.1"
)
result = subprocess.run(
    [sys.executable, "-c",
     "import torch; print(f'{torch.__version__}, CUDA: {torch.cuda.is_available()}')"],
    capture_output=True, text=True
)
print(f"   PyTorch: {result.stdout.strip()}")

# ----------------------------------------------------------
# [3/6] Remove Colab packages compiled against NumPy 2.x
#   These have C extensions with NumPy 2.x ABI (dtype size 96)
#   but NumPy 1.26.4 has dtype size 88 -- binary incompatible
# ----------------------------------------------------------
print("\n" + "=" * 60)
print("[3/6] Removing NumPy 2.x-compiled packages")
print("   pandas/scipy/numba were compiled against NumPy 2.x (dtype=96 bytes)")
print("   NumPy 1.26.4 has dtype=88 bytes -- must reinstall from scratch")
run_cmd(
    f"{sys.executable} -m pip uninstall -y pandas scipy numba",
    "Removing incompatible binaries"
)

# ----------------------------------------------------------
# [4/6] Install Chai-1 --no-deps + explicit dependencies
#   --no-deps bypasses the torch<2.7 pin in chai_lab==0.6.1
# ----------------------------------------------------------
print("\n" + "=" * 60)
print("[4/6] Installing Chai-1 and dependencies")
print("   --no-deps to bypass torch<2.7 pin (devs confirmed 2.7 works)")
if not run_cmd(
    f"{sys.executable} -m pip install 'chai_lab==0.6.1' --no-deps",
    "chai_lab==0.6.1 --no-deps"
):
    print("   Chai-1 installation failed!")
    sys.exit(1)

# Explicit deps (from chai-lab requirements.in, minus torch/numpy)
chai_deps = (
    "typer>=0.12 gemmi>=0.6.3 rdkit>=2024.9.5 "
    "biopython>=1.83 antipickle==0.2.0 tmtools>=0.0.3 "
    "modelcif>=1.0 'pandas[parquet]>=2.1,<2.3' pandera>=0.24 "
    "numba>=0.59 einops>=0.8 jaxtyping>=0.2.25 "
    "beartype>=0.18 matplotlib tqdm>=4.66 "
    "typing-extensions scipy"
)
if not run_cmd(
    f"{sys.executable} -m pip install {chai_deps}",
    "Chai-1 dependencies"
):
    print("   WARNING: Some dependencies may have failed")

# Model weight cache
os.environ['CHAI_DOWNLOADS_DIR'] = '/content/chai_weights'
os.makedirs('/content/chai_weights', exist_ok=True)

# ----------------------------------------------------------
# [5/6] Verification
# ----------------------------------------------------------
print("\n" + "=" * 60)
# Step 6: Install ipSAE_batch for interface analysis
print("\n" + "=" * 60)
print("[6/6] ipSAE_batch (interface scoring and visualization)")
run_cmd(
    f"{sys.executable} -m pip install seaborn pycirclize python-igraph plotly",
    "Installing visualization dependencies"
)
if not run_cmd(
    f"{sys.executable} -m pip install git+https://github.com/JKourelis/ipSAE_batch.git",
    "Installing ipSAE_batch"
):
    print("WARNING: ipSAE_batch failed to install. Interface analysis will be unavailable.")

    print("[5/6] Verification")
print("=" * 60)

# Verify chai_lab import
print("\n   [Testing chai_lab import]")
result = subprocess.run(
    [sys.executable, "-c",
     "from chai_lab.chai1 import run_inference; print('OK')"],
    capture_output=True, text=True
)
if result.returncode == 0 and 'OK' in result.stdout:
    print("   chai_lab import: OK")
else:
    print(f"   chai_lab import FAILED: {result.stderr[:300]}")
    result2 = subprocess.run(
        [sys.executable, "-c", "import chai_lab; print(f'v{chai_lab.__version__}')"],
        capture_output=True, text=True
    )
    if result2.returncode == 0:
        print(f"   chai_lab basic import: {result2.stdout.strip()}")
    else:
        print(f"   chai_lab completely broken: {result2.stderr[:300]}")
        sys.exit(1)

# Verify CLI
print("\n   [Testing chai-lab CLI]")
result = subprocess.run(["chai-lab", "--help"], capture_output=True, text=True)
if result.returncode == 0:
    print("   chai-lab CLI: OK")
else:
    print("   chai-lab CLI not found (will use Python API)")

# GPU bfloat16 check
print("\n   [GPU bfloat16 check]")
result = subprocess.run(
    [sys.executable, "-c",
     "import torch; "
     "cc = torch.cuda.get_device_capability(); "
     "ok = cc[0] >= 8; "
     "print(f'Compute capability: {cc[0]}.{cc[1]}, bfloat16: {ok}')"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f"   {result.stdout.strip()}")
    if 'bfloat16: False' in result.stdout:
        print()
        print("   " + "!" * 50)
        print("   WARNING: Your GPU does NOT support bfloat16!")
        print("   Chai-1 predictions will FAIL on this GPU.")
        print("   Switch to A100, L4, or H100 in Colab settings.")
        print("   " + "!" * 50)

# Test ipSAE_batch
result = subprocess.run(["ipsae-batch", "--help"], capture_output=True, text=True)
if result.returncode == 0:
    print("   ipsae-batch CLI: OK")
else:
    print("   ipsae-batch CLI: NOT AVAILABLE (interface analysis will be skipped)")

# ============================================================
# ENVIRONMENT & VERSION LOG
# ============================================================
env_lines = []
env_lines.append("\n" + "=" * 60)
env_lines.append("ENVIRONMENT & INSTALLED VERSIONS")
env_lines.append("=" * 60)

# Python version
env_lines.append(f"\nPython: {sys.version.split()[0]}")

# GPU info
gpu_result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    capture_output=True, text=True
)
if gpu_result.returncode == 0:
    env_lines.append(f"GPU: {gpu_result.stdout.strip()}")
cuda_result = subprocess.run(
    ['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'],
    capture_output=True, text=True
)
if cuda_result.returncode == 0:
    env_lines.append(f"CUDA driver: {cuda_result.stdout.strip()}")

# PyTorch + CUDA runtime
result = subprocess.run(
    [sys.executable, "-c",
     "import torch; print(f'PyTorch: {torch.__version__}'); "
     "print(f'CUDA available: {torch.cuda.is_available()}'); "
     "print(f'CUDA runtime: {torch.version.cuda}') if torch.cuda.is_available() else None"],
    capture_output=True, text=True
)
if result.returncode == 0:
    for line in result.stdout.strip().split('\n'):
        env_lines.append(f"   {line}")

# All relevant package versions via pip
result = subprocess.run(
    [sys.executable, "-m", "pip", "list", "--format=freeze"],
    capture_output=True, text=True
)
all_packages = result.stdout.strip().split('\n')

# Chai-1 + core deps
env_lines.append("\nChai-1 stack:")
tool_pkgs = [
    'chai-lab',
    'numpy',
    'torch',
    'torchvision',
    'rdkit',
    'gemmi',
    'numba',
    'beartype',
    'jaxtyping',
    'einops',
    'pandas',
    'scipy',
    'biopython',
    'typer',
    'tmtools',
    'modelcif',
    'pandera',
    'tqdm',
    'matplotlib',
]
for pkg in tool_pkgs:
    pkg_lower = pkg.lower().replace('-', '_')
    for line in all_packages:
        line_lower = line.lower().replace('-', '_')
        if line_lower.startswith(pkg_lower + '=='):
            env_lines.append(f"   {line}")
            break
    else:
        # Try partial match
        for line in all_packages:
            if pkg_lower in line.lower().replace('-', '_'):
                env_lines.append(f"   {line}")
                break

# ipSAE_batch + visualization deps
env_lines.append("\nipSAE_batch stack:")
ipsae_pkgs = [
    'ipsae-batch', 'seaborn', 'pycirclize', 'python-igraph', 'plotly',
    'matplotlib', 'networkx',
]
for pkg in ipsae_pkgs:
    pkg_lower = pkg.lower().replace('-', '_')
    for line in all_packages:
        line_lower = line.lower().replace('-', '_')
        if line_lower.startswith(pkg_lower + '=='):
            env_lines.append(f"   {line}")
            break
    else:
        for line in all_packages:
            if pkg_lower in line.lower().replace('-', '_'):
                env_lines.append(f"   {line}")
                break
        else:
            env_lines.append(f"   {pkg}: not installed")


# Cleanup and mark ready
if os.path.exists(restart_marker):
    os.remove(restart_marker)
with open(ready_marker, "w") as f:
    f.write("Ready")

env_lines.append("\n" + "=" * 60)

# Write environment log to file and print
env_log = "\n".join(str(x) for x in env_lines)
print(env_log)
with open("/content/environment.txt", "w") as _ef:
    _ef.write(env_log + "\n")
print("CHAI-1 INSTALLATION COMPLETE")
print("=" * 60)
print("Next: Run Cell 3 to set up the CSV processor")

# ============================================================
# PAE PATCH: Inject PAE saving into chai_lab/chai1.py
# ============================================================
print("\n[Applying PAE patch to chai_lab...]")
import importlib.util
import re

chai_spec = importlib.util.find_spec("chai_lab")
if chai_spec is not None:
    chai1_py = os.path.join(os.path.dirname(chai_spec.origin), "chai1.py")
    if os.path.exists(chai1_py):
        with open(chai1_py, "r") as f:
            chai1_src = f.read()
        old_line = "        np.savez(scores_out_path, **get_scores(ranking_outputs))"
        new_lines = (
            "        np.savez(scores_out_path, **get_scores(ranking_outputs))\n"
            "        pae_out_path = output_dir.joinpath(f\"pae.model_idx_{idx}.npz\")\n"
            "        np.savez(pae_out_path, pae=pae_scores[idx].numpy())"
        )
        if old_line in chai1_src:
            patched_src = chai1_src.replace(old_line, new_lines)
            with open(chai1_py, "w") as f:
                f.write(patched_src)
            print(f"   PAE patch applied to: {chai1_py}")
        elif "pae_out_path" in chai1_src:
            print("   PAE patch already applied — skipping.")
        else:
            print(f"   WARNING: PAE patch target not found in {chai1_py}")
            print("   The chai_lab version may have changed.")
    else:
        print(f"   WARNING: chai1.py not found at expected path: {chai1_py}")
else:
    print("   WARNING: chai_lab module not found — skipping PAE patch.")


In [ ]:
#@title Cell 4: Chai-1 CSV Processor Setup
import pandas as pd
import os
import re
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
from io import StringIO

class ChaiJobProcessor:
    """Process CSV files for Chai-1 batch predictions.

    Converts the unified CSV format into Chai-1's extended FASTA format:
      >protein|chain_name
      SEQUENCE...

      >ligand|ligand_name
      SMILES_STRING

      >dna|dna_name
      SEQUENCE...

      >rna|rna_name
      SEQUENCE...
    """

    # ================================================================
    # LIGAND CCD -> SMILES LOOKUP TABLE
    # ================================================================
    LIGAND_SMILES = {
        # Nucleotides
        'ATP': 'c1nc(c2c(n1)n(cn2)[C@@H]3[C@@H]([C@@H]([C@H](O3)COP(=O)(O)OP(=O)(O)OP(=O)(O)O)O)O)N',
        'ADP': 'c1nc(c2c(n1)n(cn2)[C@@H]3[C@@H]([C@@H]([C@H](O3)COP(=O)(O)OP(=O)(O)O)O)O)N',
        'AMP': 'c1nc(c2c(n1)n(cn2)[C@@H]3[C@@H]([C@@H]([C@H](O3)COP(=O)(O)O)O)O)N',
        'GTP': 'c1[nH]c2c(n1)c(=O)[nH]c(n2)N.[C@@H]3([C@@H]([C@@H]([C@H](O3)COP(=O)(O)OP(=O)(O)OP(=O)(O)O)O)O)',
        'GDP': 'c1[nH]c2c(n1)c(=O)[nH]c(n2)N.[C@@H]3([C@@H]([C@@H]([C@H](O3)COP(=O)(O)OP(=O)(O)O)O)O)',
        'GMP': 'c1[nH]c2c(n1)c(=O)[nH]c(n2)N.[C@@H]3([C@@H]([C@@H]([C@H](O3)COP(=O)(O)O)O)O)',
        'CTP': 'c1cn(c(=O)nc1N)[C@@H]2[C@@H]([C@@H]([C@H](O2)COP(=O)(O)OP(=O)(O)OP(=O)(O)O)O)O',
        'CDP': 'c1cn(c(=O)nc1N)[C@@H]2[C@@H]([C@@H]([C@H](O2)COP(=O)(O)OP(=O)(O)O)O)O',
        'UTP': 'c1cn(c(=O)[nH]c1=O)[C@@H]2[C@@H]([C@@H]([C@H](O2)COP(=O)(O)OP(=O)(O)OP(=O)(O)O)O)O',
        'UDP': 'c1cn(c(=O)[nH]c1=O)[C@@H]2[C@@H]([C@@H]([C@H](O2)COP(=O)(O)OP(=O)(O)O)O)O',
        # Cofactors
        'NAD': 'c1cc(c[n+](c1)[C@@H]2[C@@H]([C@@H]([C@H](O2)COP(=O)([O-])OP(=O)(O)OC[C@@H]3[C@H]([C@H]([C@@H](O3)n4cnc5c4ncnc5N)O)O)O)O)C(=O)N',
        'NAP': 'c1cc(c[n+](c1)[C@@H]2[C@@H]([C@@H]([C@H](O2)COP(=O)([O-])OP(=O)(O)OC[C@@H]3[C@H]([C@H]([C@@H](O3)n4cnc5c4ncnc5N)OP(=O)(O)O)O)O)O)C(=O)N',
        'FAD': 'Cc1cc2c(cc1C)N(C3=NC(=O)NC(=O)C3=N2)C[C@@H]([C@@H]([C@@H](COP(=O)(O)OP(=O)(O)OC[C@@H]4[C@H]([C@H]([C@@H](O4)n5cnc6c5ncnc6N)O)O)O)O)O',
        'FMN': 'Cc1cc2c(cc1C)N(C3=NC(=O)NC(=O)C3=N2)C[C@@H]([C@@H]([C@@H](COP(=O)(O)O)O)O)O',
        'COA': 'CC(C)(COP(=O)(O)OP(=O)(O)OC[C@@H]1[C@H]([C@H]([C@@H](O1)n2cnc3c2ncnc3N)O)OP(=O)(O)O)[C@@H](C(=O)NCCC(=O)NCCS)O',
        'ACO': 'CC(=O)SCCNC(=O)CCNC(=O)[C@@H](C(C)(C)COP(=O)(O)OP(=O)(O)OC[C@@H]1[C@H]([C@H]([C@@H](O1)n2cnc3c2ncnc3N)O)OP(=O)(O)O)O',
        # Vitamins / cofactors
        'PLP': 'Cc1c(c(c(cn1)COP(=O)(O)O)C=O)O',
        'TPP': 'Cc1c(c(=O)n(c(n1)N)Cc2c(ccn2CC(COP(=O)(O)OP(=O)(O)O)C)C)C',
        'BTN': 'C1[C@H]2[C@@H](S1)CCCCC(=O)N[C@@H]2C(=O)O',
        'SAH': 'c1nc(c2c(n1)n(cn2)[C@@H]3[C@@H]([C@@H]([C@H](O3)CSCC[C@@H](C(=O)O)N)O)O)N',
        'SAM': 'c1nc(c2c(n1)n(cn2)[C@@H]3[C@@H]([C@@H]([C@H](O3)CS(CC[C@@H](C(=O)O)N)C)O)O)N',
        # Heme
        'HEM': 'C=CC1=C(C2=Cc3c(c([nH]3)C=C4C(=C(C(=N4)C=c5c(c(c([nH]5)C=C1[N-]2)C=C)C)CCC(=O)O)C)CCC(=O)O)C.[Fe+2]',
        # Other common ligands
        'MTA': 'c1nc(c2c(n1)n(cn2)[C@@H]3[C@@H]([C@@H]([C@H](O3)CSC)O)O)N',
        'THM': 'Cc1c(c(=O)n(c(n1)N)Cc2c(ccn2CC(C)O)C)C',
        # Sugars / carbohydrates
        'NAG': 'CC(=O)N[C@@H]1[C@H]([C@@H]([C@H](OC1O)CO)O)O',
        'MAN': 'OC[C@H]1OC(O)[C@@H](O)[C@@H](O)[C@@H]1O',
        'FUC': 'C[C@H]1[C@H]([C@H]([C@@H]([C@@H](O1)O)O)O)O',
        'GAL': 'OC[C@H]1OC(O)[C@H](O)[C@@H](O)[C@H]1O',
        'GLC': 'OC[C@H]1OC(O)[C@H](O)[C@@H](O)[C@@H]1O',
        'BMA': 'OC[C@H]1OC(O)[C@@H](O)[C@@H](O)[C@@H]1O',
        'SIA': 'CC(=O)N[C@@H]1[C@@H](C[C@@](O[C@H]1[C@@H]([C@@H](CO)O)O)(C(=O)O)O)O',
    }

    # ================================================================
    # ION CCD -> SMILES LOOKUP TABLE
    # ================================================================
    ION_SMILES = {
        'MG': '[Mg+2]',
        'ZN': '[Zn+2]',
        'CA': '[Ca+2]',
        'FE': '[Fe+2]',
        'FE2': '[Fe+2]',
        'FE3': '[Fe+3]',
        'MN': '[Mn+2]',
        'CU': '[Cu+2]',
        'CO': '[Co+2]',
        'NI': '[Ni+2]',
        'K': '[K+]',
        'NA': '[Na+]',
        'CL': '[Cl-]',
        'CD': '[Cd+2]',
        'PB': '[Pb+2]',
        'SR': '[Sr+2]',
        'BA': '[Ba+2]',
        'LI': '[Li+]',
        'CS': '[Cs+]',
        'RB': '[Rb+]',
    }

    # ================================================================
    # REFERENCE DATA (heavy atom counts for token estimation)
    # ================================================================
    EMBEDDED_REFERENCE = """Type,CCD Code,Name,Target Residue,Molecular Formula,All Atom Count,Heavy Atom Count
PTM,SEP,Phosphoserine,SER,C3H8NO6P,18,11
PTM,TPO,Phosphothreonine,THR,C4H10NO6P,21,12
PTM,PTR,Phosphotyrosine,TYR,C9H12NO6P,28,17
PTM,MLY,N-Methyllysine,LYS,C7H16N2O2,27,11
PTM,ALY,N-Acetyllysine,LYS,C8H16N2O3,29,13
PTM,HYP,Hydroxyproline,PRO,C5H9NO3,18,9
PTM,M3L,N-Trimethyllysine,LYS,C9H20N2O2,33,13
PTM,MLZ,N-Methyllysine,LYS,C7H16N2O2,27,11
PTM,CSD,Cysteine sulfinic acid,CYS,C3H7NO4S,16,9
PTM,CSO,S-Hydroxycysteine,CYS,C3H7NO3S,15,8
PTM,CGU,Gamma-carboxyglutamic acid,GLU,C5H7NO6,21,12
PTM,FME,N-Formylmethionine,MET,C6H11NO3S,22,11
PTM,NEP,N-(phosphonoethyl)isoleucine,ILE,C8H18NO5P,32,15
PTM,HIC,4-Methyl-histidine,HIS,C7H11N3O2,23,12
PTM,CAS,S-(dimethylarsenic)cysteine,CYS,C5H11AsNO2S,20,9
Ligand,ATP,Adenosine triphosphate,NA,C10H16N5O13P3,47,31
Ligand,ADP,Adenosine diphosphate,NA,C10H15N5O10P2,42,27
Ligand,AMP,Adenosine monophosphate,NA,C10H14N5O7P,37,23
Ligand,GTP,Guanosine triphosphate,NA,C10H16N5O14P3,48,32
Ligand,GDP,Guanosine diphosphate,NA,C10H15N5O11P2,43,28
Ligand,GMP,Guanosine monophosphate,NA,C10H14N5O8P,38,24
Ligand,CTP,Cytidine triphosphate,NA,C9H16N3O14P3,45,29
Ligand,CDP,Cytidine diphosphate,NA,C9H15N3O11P2,40,25
Ligand,UTP,Uridine triphosphate,NA,C9H15N2O15P3,44,29
Ligand,UDP,Uridine diphosphate,NA,C9H14N2O12P2,39,25
Ligand,NAD,Nicotinamide adenine dinucleotide,NA,C21H27N7O14P2,71,44
Ligand,NAP,NADP,NA,C21H28N7O17P3,86,55
Ligand,FAD,Flavin adenine dinucleotide,NA,C27H33N9O15P2,91,53
Ligand,FMN,Flavin mononucleotide,NA,C17H21N4O9P,52,31
Ligand,HEM,Heme,NA,C34H32FeN4O4,75,43
Ligand,SAH,S-Adenosyl-L-homocysteine,NA,C14H20N6O5S,46,26
Ligand,SAM,S-Adenosyl-L-methionine,NA,C15H22N6O5S,49,27
Ligand,COA,Coenzyme A,NA,C21H36N7O16P3S,90,57
Ligand,ACO,Acetyl coenzyme A,NA,C23H38N7O17P3S,99,61
Ligand,PLP,Pyridoxal-5-phosphate,NA,C8H10NO6P,25,16
Ligand,TPP,Thiamine diphosphate,NA,C12H19N4O7P2S,45,25
Ligand,BTN,Biotin,NA,C10H16N2O3S,32,16
Ligand,MTA,5-Methylthioadenosine,NA,C11H15N5O3S,35,20
Ligand,THM,Thiamine,NA,C12H17ClN4OS,38,18
Ion,MG,Magnesium ion,NA,Mg,1,1
Ion,ZN,Zinc ion,NA,Zn,1,1
Ion,CA,Calcium ion,NA,Ca,1,1
Ion,FE,Iron ion,NA,Fe,1,1
Ion,MN,Manganese ion,NA,Mn,1,1
Ion,CU,Copper ion,NA,Cu,1,1
Ion,CO,Cobalt ion,NA,Co,1,1
Ion,NI,Nickel ion,NA,Ni,1,1
Ion,K,Potassium ion,NA,K,1,1
Ion,NA,Sodium ion,NA,Na,1,1
Ion,CL,Chloride ion,NA,Cl,1,1
Glycan,NAG,N-Acetyl-D-glucosamine,NA,C8H15NO6,30,15
Glycan,MAN,D-Mannose,NA,C6H12O6,24,12
Glycan,FUC,L-Fucose,NA,C6H12O5,23,11
Glycan,GAL,D-Galactose,NA,C6H12O6,24,12
Glycan,SIA,N-Acetylneuraminic acid,NA,C11H19NO9,40,21
Glycan,GLC,D-Glucose,NA,C6H12O6,24,12
Glycan,BMA,beta-D-Mannose,NA,C6H12O6,24,12
Glycan,NDG,N-Acetyl-D-glucosamine,NA,C8H15NO6,30,15
Glycan,A2G,N-Acetyl-D-galactosamine,NA,C8H15NO6,30,15
Glycan,FUL,L-Fucose,NA,C6H12O5,23,11"""

    def __init__(self, reference_csv: Optional[str] = None):
        """Initialize processor with reference data."""
        if reference_csv:
            self.reference_data = pd.read_csv(reference_csv)
        else:
            self.reference_data = pd.read_csv(StringIO(self.EMBEDDED_REFERENCE))

        self.ptm_lookup = self._create_lookup('PTM')
        self.ligand_lookup = self._create_lookup('Ligand')
        self.ion_lookup = self._create_lookup('Ion')
        self.glycan_lookup = self._create_lookup('Glycan')

        self.amino_acids = set('ACDEFGHIKLMNPQRSTVWY')

    def _create_lookup(self, type_name: str) -> Dict[str, Dict[str, Any]]:
        """Create lookup dictionary for a specific type."""
        type_data = self.reference_data[self.reference_data['Type'] == type_name]
        lookup = {}
        for _, row in type_data.iterrows():
            if pd.notna(row['CCD Code']):
                atom_count = 1
                if pd.notna(row.get('Heavy Atom Count')):
                    try:
                        atom_count = int(row['Heavy Atom Count'])
                    except (ValueError, TypeError):
                        atom_count = 1
                lookup[row['CCD Code']] = {
                    'name': row['Name'],
                    'target_residue': row['Target Residue'] if pd.notna(row['Target Residue']) else 'NA',
                    'atom_count': atom_count
                }
        return lookup

    def _validate_sequence_characters(self, sequence: str, seq_type: str) -> List[str]:
        """Validate sequence contains only allowed characters."""
        errors = []
        sequence = sequence.upper()

        if seq_type.lower() == 'protein':
            invalid_chars = []
            for i, char in enumerate(sequence, 1):
                if char.isalpha() and char not in self.amino_acids:
                    invalid_chars.append(f"{char}@{i}")
                elif not char.isalpha() and not char.isspace():
                    invalid_chars.append(f"{char}@{i}")
            if invalid_chars:
                errors.append(f"Invalid amino acids: {', '.join(invalid_chars[:10])}" +
                            ("..." if len(invalid_chars) > 10 else ""))

        elif seq_type.lower() == 'dna':
            valid_bases = set('ATCG')
            invalid_chars = []
            for i, char in enumerate(sequence, 1):
                if char.isalpha() and char not in valid_bases:
                    invalid_chars.append(f"{char}@{i}")
                elif not char.isalpha() and not char.isspace():
                    invalid_chars.append(f"{char}@{i}")
            if invalid_chars:
                errors.append(f"Invalid DNA bases: {', '.join(invalid_chars[:10])}" +
                            ("..." if len(invalid_chars) > 10 else ""))

        elif seq_type.lower() == 'rna':
            valid_bases = set('AUCG')
            invalid_chars = []
            for i, char in enumerate(sequence, 1):
                if char.isalpha() and char not in valid_bases:
                    invalid_chars.append(f"{char}@{i}")
                elif not char.isalpha() and not char.isspace():
                    invalid_chars.append(f"{char}@{i}")
            if invalid_chars:
                errors.append(f"Invalid RNA bases: {', '.join(invalid_chars[:10])}" +
                            ("..." if len(invalid_chars) > 10 else ""))

        return errors

    def _is_smiles(self, ligand_string: str) -> bool:
        """Check if string is likely a SMILES representation rather than a CCD code."""
        if len(ligand_string) < 2:
            return False
        # CCD codes are typically 1-3 uppercase letters/numbers
        if re.match(r'^[A-Z0-9]{1,3}$', ligand_string):
            return False
        # SMILES contain special characters
        smiles_chars = set('[]()=#@+-\\/.')
        return any(char in smiles_chars for char in ligand_string) or len(ligand_string) > 3

    def _get_ligand_smiles(self, ligand_string: str) -> Tuple[Optional[str], Optional[str]]:
        """Look up SMILES for a ligand string (CCD code or raw SMILES)."""
        ligand_upper = ligand_string.strip().upper()

        # Check known lookups FIRST (before _is_smiles which has false positives)
        if ligand_upper in self.ION_SMILES:
            return self.ION_SMILES[ligand_upper], None

        if hasattr(self, "LIGAND_SMILES") and ligand_upper in self.LIGAND_SMILES:
            return self.LIGAND_SMILES[ligand_upper], None

        if hasattr(self, "ligand_lookup") and ligand_upper in self.ligand_lookup:
            smiles = self.ligand_lookup[ligand_upper].get("smiles")
            if smiles:
                return smiles, None

        # Not a known CCD code — check if it looks like SMILES
        if self._is_smiles(ligand_string):
            return ligand_string.strip(), None

        return None, f"Unknown ligand: '{ligand_string}'. Use a CCD code or SMILES string."

    def _sanitize_jobname(self, jobname: str) -> Tuple[str, List[str]]:
        """Sanitize jobname for filesystem compatibility."""
        errors = []

        if pd.isna(jobname) or str(jobname).strip() == '':
            errors.append("Missing jobname")
            return '', errors

        original = str(jobname)
        sanitized = original.lower()
        sanitized = sanitized.replace('-', '_')
        sanitized = re.sub(r'[^a-z0-9_]', '', sanitized)

        if len(sanitized) > 128:
            sanitized = sanitized[:128]
            errors.append(f"Jobname truncated to 128 characters")

        if not sanitized.strip():
            errors.append(f"Jobname '{original}' became empty after sanitization")
            return '', errors

        return sanitized, errors

    def _process_job(self, row: pd.Series) -> Tuple[Optional[Dict], List[str]]:
        """Process a single job row from CSV."""
        errors = []
        warnings = []
        all_sequences = []

        jobname, jobname_errors = self._sanitize_jobname(row.get('jobname', ''))
        errors.extend(jobname_errors)

        if not jobname:
            return None, errors

        chain_counter = 0

        for i in range(1, 11):
            name_col = f'seq{i}_name'
            type_col = f'seq{i}_type'
            copies_col = f'seq{i}_copies'
            seq_col = f'seq{i}'
            mods_col = f'seq{i}_mods'

            if seq_col not in row:
                continue

            user_name = row.get(name_col, '')
            seq_type = row.get(type_col, '')
            seq_type = str(seq_type).strip().lower()
            copies = row.get(copies_col, 1)
            sequence = row.get(seq_col, '')
            mods = row.get(mods_col, '')

            if pd.isna(sequence) or str(sequence).strip() == '':
                continue
            if pd.isna(seq_type) or str(seq_type).strip() == '':
                continue

            sequence = str(sequence).strip()
            seq_type = str(seq_type).strip().lower()
            copies = int(copies) if pd.notna(copies) and str(copies).strip() != '' else 1
            user_name = str(user_name).strip() if pd.notna(user_name) and str(user_name).strip() != '' else f'chain_{chain_counter}'

            # Check for modifications (Chai-1 warning)
            if pd.notna(mods) and str(mods).strip() != '':
                warnings.append(f"Modifications for {user_name} will be IGNORED: "
                              f"Chai-1 does not support PTMs in its FASTA format")

            if seq_type in ['protein', 'dna', 'rna']:
                char_errors = self._validate_sequence_characters(sequence, seq_type)
                errors.extend(char_errors)

                for copy_idx in range(copies):
                    chain_name = user_name if copies == 1 else f"{user_name}_{copy_idx+1}"
                    all_sequences.append({
                        'type': seq_type,
                        'name': chain_name,
                        'sequence': sequence.upper(),
                    })
                    chain_counter += 1

            elif seq_type == 'ligand':
                ligand_string = sequence.strip()
                smiles, smiles_error = self._get_ligand_smiles(ligand_string)

                if smiles_error:
                    errors.append(smiles_error)
                    continue

                for copy_idx in range(copies):
                    chain_name = user_name if copies == 1 else f"{user_name}_{copy_idx+1}"
                    all_sequences.append({
                        'type': 'ligand',
                        'name': chain_name,
                        'smiles': smiles,
                        'original_code': ligand_string,
                    })
                    chain_counter += 1

            else:
                errors.append(f"Unsupported sequence type: '{seq_type}' for {user_name}")

        if not all_sequences:
            errors.append("No valid sequences found")
            return None, errors

        # Add warnings as non-fatal errors
        errors.extend(warnings)

        job = {
            'name': jobname,
            'sequences': all_sequences,
            'has_modifications': any('IGNORED' in e for e in warnings),
            'has_pocket': False,  # Chai-1 doesn't support pocket constraints
            'has_covalent': False,  # Chai-1 doesn't support covalent bond constraints
        }

        # Check for pocket/covalent constraints and warn
        pocket_binder = row.get('pocket_binder', '')
        if pd.notna(pocket_binder) and str(pocket_binder).strip() != '':
            errors.append("Pocket constraints are not supported by Chai-1 (ignored)")

        covalent_bonds = row.get('covalent_bonds', '')
        if pd.notna(covalent_bonds) and str(covalent_bonds).strip() != '':
            errors.append("Covalent bond constraints are not supported by Chai-1 (ignored)")

        return job, errors

    def _generate_fasta(self, job: Dict) -> str:
        """Generate Chai-1 extended FASTA format.

        Format:
            >protein|chain_name
            SEQUENCE

            >ligand|ligand_name
            SMILES

            >dna|dna_name
            SEQUENCE

            >rna|rna_name
            SEQUENCE
        """
        lines = []
        for seq in job['sequences']:
            seq_type = seq['type']
            seq_name = seq['name']

            if seq_type == 'protein':
                lines.append(f">protein|name={seq_name}")
                lines.append(seq['sequence'])
            elif seq_type == 'dna':
                lines.append(f">dna|name={seq_name}")
                lines.append(seq['sequence'])
            elif seq_type == 'rna':
                lines.append(f">rna|name={seq_name}")
                lines.append(seq['sequence'])
            elif seq_type == 'ligand':
                lines.append(f">ligand|name={seq_name}")
                lines.append(seq['smiles'])

        return '\n'.join(lines) + '\n'

    def process_csv(self, csv_path: str) -> Tuple[List[Dict], pd.DataFrame]:
        """Process CSV file and return jobs list and summary DataFrame."""
        df = pd.read_csv(csv_path)

        jobs = []
        summary_rows = []

        for idx, row in df.iterrows():
            job, errors = self._process_job(row)

            # Separate warnings from real errors
            real_errors = [e for e in errors if 'IGNORED' not in e and 'not supported' not in e]
            warnings = [e for e in errors if 'IGNORED' in e or 'not supported' in e]

            if job:
                jobs.append(job)
                status = 'WARNING' if warnings and not real_errors else ('ERROR' if real_errors else 'OK')
                summary_rows.append({
                    'jobname': job['name'],
                    'sequences': len(job['sequences']),
                    'has_modifications': job['has_modifications'],
                    'has_pocket': job['has_pocket'],
                    'has_covalent': job['has_covalent'],
                    'status': status,
                    'errors': '; '.join(errors) if errors else ''
                })
            else:
                summary_rows.append({
                    'jobname': f"Row {idx+1}",
                    'sequences': 0,
                    'has_modifications': False,
                    'has_pocket': False,
                    'has_covalent': False,
                    'status': 'FAILED',
                    'errors': '; '.join(errors)
                })

        summary_df = pd.DataFrame(summary_rows)
        return jobs, summary_df

# ================================================================
# Initialize processor
# ================================================================
print("Initializing Chai-1 CSV Processor...")
chai_processor = ChaiJobProcessor()

print("Using embedded reference data")
print("Processor ready")
print(f"SMILES lookup: {len(ChaiJobProcessor.LIGAND_SMILES)} ligands, {len(ChaiJobProcessor.ION_SMILES)} ions")
print(f"Reference data: {len(chai_processor.ligand_lookup)} ligands, {len(chai_processor.ion_lookup)} ions, {len(chai_processor.glycan_lookup)} glycans")
print()
print("Supported ligand CCD codes (auto-converted to SMILES):")
print(f"   Nucleotides: ATP, ADP, AMP, GTP, GDP, GMP, CTP, CDP, UTP, UDP")
print(f"   Cofactors: NAD, NAP(NADP), FAD, FMN, COA, ACO, PLP, TPP, BTN, SAH, SAM")
print(f"   Heme: HEM")
print(f"   Sugars: NAG, MAN, FUC, GAL, GLC, BMA, SIA")
print(f"   Ions: MG, ZN, CA, FE, MN, CU, CO, NI, K, NA, CL")
print()
print("NOTE: Chai-1 does NOT support:")
print("   - Post-translational modifications (PTMs) in FASTA format")
print("   - Pocket binding constraints")
print("   - Covalent bond constraints")
print("   These will be warned about but jobs will still run without them.")
print()
# CSV processing follows below

# ============================================================
# PROCESS UPLOADED CSV
# ============================================================
import pandas as pd

if 'global_settings' not in globals() or 'csv_filename' not in global_settings:
    print("Error: Please run Cell 2 (Upload CSV) first")
    raise ValueError("No CSV uploaded")

csv_filename = global_settings['csv_filename']

print("\nProcessing CSV...")
try:
    jobs, summary_df = chai_processor.process_csv(csv_filename)
except Exception as e:
    print(f"CSV processing failed: {e}")
    raise

print("\n" + "=" * 60)
print("JOB SUMMARY")
print("=" * 60)
print(summary_df.to_string(index=False))

if jobs:
    print("\n" + "=" * 60)
    print(f"FASTA PREVIEW (first job: {jobs[0]['name']})")
    print("=" * 60)
    fasta_preview = chai_processor._generate_fasta(jobs[0])
    print(fasta_preview)

error_jobs = summary_df[summary_df['status'] == 'FAILED']
warning_jobs = summary_df[summary_df['status'] == 'WARNING']

if len(warning_jobs) > 0:
    print(f"\n{len(warning_jobs)} jobs have warnings (will still run):")
    for _, row in warning_jobs.iterrows():
        print(f"  - {row['jobname']}: {row['errors']}")

if len(error_jobs) > 0:
    print(f"\n{len(error_jobs)} jobs FAILED validation:")
    for _, row in error_jobs.iterrows():
        print(f"  - {row['jobname']}: {row['errors']}")

    if len(jobs) > 0:
        proceed = input("\nProceed with valid jobs only? (yes/no): ").strip().lower()
        if proceed not in ['yes', 'y']:
            raise ValueError("Processing cancelled by user")
    else:
        raise ValueError("No valid jobs found in CSV")

global_settings.update({
    'batch_jobs': jobs,
    'processor': chai_processor,
})

print("\n" + "=" * 60)
print(f"READY TO PROCESS {len(jobs)} JOBS")
print("=" * 60)
if global_settings.get('msa_folder'):
    print(f"Pre-computed MSAs: {global_settings['msa_folder']}")

print("\nNext: Configure settings (Cell 5), then run (Cell 6)")

In [ ]:
#@title Cell 5: Configure Chai-1 Prediction Parameters

print("=" * 60)
print("CHAI-1 PREDICTION PARAMETERS")
print("=" * 60)

# Core Model Parameters
num_trunk_recycles = 3 #@param {type:"integer"}
#@markdown - **Trunk recycles**: Number of recycling iterations through the model trunk. More recycles improve accuracy but increase runtime. **Time**: ~Linear scaling. **VRAM**: Minimal impact.

num_diffn_timesteps = 200 #@param {type:"integer"}
#@markdown - **Diffusion timesteps**: Number of denoising steps in the diffusion process. More steps = higher quality structures. **Time**: Linear scaling (50 = 4x faster than 200). **VRAM**: Moderate impact.

seed = 42 #@param {type:"integer"}
#@markdown - **Random seed**: Controls reproducibility. Use different seeds for structural diversity.

# MSA Settings
#@markdown > **Note**: These MSA settings only apply when `msa_folder` is empty (Cell 4).
#@markdown > When pre-computed MSAs are provided, online MSA search is skipped entirely.

use_msa_server = True #@param {type:"boolean"}
#@markdown - **Use MSA server**: Automatically generate MSAs using ColabFold's MMseqs2 server. Recommended for best accuracy. **Time**: Adds 30-120 seconds per protein sequence. **VRAM**: No impact.

msa_server_url = "https://api.colabfold.com" #@param {type:"string"}
#@markdown - **MSA server URL**: ColabFold server endpoint for MSA generation. Default is the public ColabFold API.

#@markdown ---
# ============================================================
# Sampling Settings
# ============================================================
num_diffn_samples = 5 #@param {type:"integer"}
#@markdown - **Diffusion samples**: Number of independent structure predictions per input. More samples improve reliability. **Time**: Linear scaling. **VRAM**: Linear scaling. Total models = trunk_samples × diffn_samples.

num_trunk_samples = 1 #@param {type:"integer"}
#@markdown - **Trunk samples**: Independent passes through the model trunk. Increases structural diversity. Total models = trunk_samples × diffn_samples. **Time**: Linear scaling. **VRAM**: No additional per-sample.

#@markdown ---
# ============================================================
# Advanced Settings
# ============================================================
use_esm_embeddings = True #@param {type:"boolean"}
#@markdown - **ESM embeddings**: Use ESM protein language model features. Improves accuracy but adds ~30s per protein. Disable to speed up at slight cost of quality. **Time**: +30s/protein. **VRAM**: +1-2GB.

low_memory = True #@param {type:"boolean"}
#@markdown - **Low memory mode**: CPU offloading for GPUs < 80GB VRAM. Keep enabled unless using A100-80GB. **Time**: +10-20%. **VRAM**: Saves ~10-15GB.

# Output Settings
#@markdown **Output**: Chai-1 always outputs mmCIF format (.cif files). No other format is supported.

skip_existing = False #@param {type:"boolean"}
#@markdown - **Skip existing**: If True, skip predictions where output files already exist.

#@markdown ---
#@markdown ### Interface Analysis (ipSAE_batch)
#@markdown Scores interface quality (ipSAE, ipTM, pDockQ) and generates visualizations for each prediction.

enable_ipsae = True #@param {type:"boolean"}
#@markdown - **Enable**: Run ipSAE interface analysis on each completed job.

ipsae_png = True #@param {type:"boolean"}
#@markdown - **PNG plots**: Matrix + ribbon visualizations per model (~10-30s/job).

ipsae_pdf = False #@param {type:"boolean"}
#@markdown - **PDF report**: Side-by-side model comparison per job (~20-60s/job).

ipsae_per_residue = False #@param {type:"boolean"}
#@markdown - **Per-residue CSV**: Detailed per-residue interface scores.

ipsae_per_contact = False #@param {type:"boolean"}
#@markdown - **Per-contact CSV**: Per-contact-pair scores (most detailed).

ipsae_pae_cutoff = 10.0 #@param {type:"number"}
#@markdown - **PAE cutoff**: PAE threshold for interface scoring (default: 10.0).

ipsae_dist_cutoff = 10.0 #@param {type:"number"}
#@markdown - **Distance cutoff**: CB-CB distance threshold in Angstroms (default: 10.0).

ipsae_select_residues = "" #@param {type:"string"}
#@markdown - **Select residues**: Focus on specific residues, e.g. `A:100-105,B:203`. Empty = all interfaces.

ipsae_batch_analysis = True #@param {type:"boolean"}
#@markdown - **Final batch comparison**: Combined CSV + interactive HTML across all jobs.

# Check if global_settings exists
if 'global_settings' not in globals():
    print("\nError: Please run Cell 4 (CSV Upload) first")
    raise ValueError("No batch jobs configured")

# Store all settings
global_settings['config'] = {
    'num_trunk_recycles': num_trunk_recycles,
    'num_diffn_timesteps': num_diffn_timesteps,
    'seed': seed,
    'use_msa_server': use_msa_server,
    'msa_server_url': msa_server_url,
    'num_diffn_samples': num_diffn_samples,
    'num_trunk_samples': num_trunk_samples,
    'use_esm_embeddings': use_esm_embeddings,
    'low_memory': low_memory,
    'skip_existing': skip_existing,
    'enable_ipsae': enable_ipsae,
    'ipsae_png': ipsae_png,
    'ipsae_pdf': ipsae_pdf,
    'ipsae_per_residue': ipsae_per_residue,
    'ipsae_per_contact': ipsae_per_contact,
    'ipsae_pae_cutoff': ipsae_pae_cutoff,
    'ipsae_dist_cutoff': ipsae_dist_cutoff,
    'ipsae_select_residues': ipsae_select_residues,
    'ipsae_batch_analysis': ipsae_batch_analysis,
}

# Display summary
print("\nCONFIGURATION SUMMARY")
print("=" * 60)
print(f"Model Settings:")
print(f"  - Trunk recycles: {num_trunk_recycles}")
print(f"  - Diffusion timesteps: {num_diffn_timesteps}")
print(f"  - Random seed: {seed}")
print(f"\nMSA Generation:")
print(f"  - Use MSA server: {use_msa_server}")
print(f"  - Server URL: {msa_server_url}")

print(f"\nSampling:")
print(f"  - Diffusion samples: {num_diffn_samples}")
print(f"  - Trunk samples: {num_trunk_samples}")
print(f"  - Total models per job: {num_trunk_samples * num_diffn_samples}")
print(f"\nAdvanced:")
print(f"  - ESM embeddings: {use_esm_embeddings}")
print(f"  - Low memory mode: {low_memory}")
print(f"\nOutput:")
print(f"  - Skip existing: {skip_existing}")


print(f"\nInterface Analysis (ipSAE):")
print(f"  - Enabled: {enable_ipsae}")
if enable_ipsae:
    print(f"  - PNG: {ipsae_png} | PDF: {ipsae_pdf}")
    print(f"  - Per-residue: {ipsae_per_residue} | Per-contact: {ipsae_per_contact}")
    print(f"  - Cutoffs: PAE={ipsae_pae_cutoff}, dist={ipsae_dist_cutoff}")
    if ipsae_select_residues:
        print(f"  - Selected residues: {ipsae_select_residues}")
    print(f"  - Final batch comparison: {ipsae_batch_analysis}")

print(f"\nNext: Run Cell 6 to start batch predictions")


In [ ]:
#@title Cell 6: Run Batch Predictions with Calibration-Based Parallel GPU Scheduling
#@markdown **Runs job 1 alone to calibrate VRAM usage, then launches remaining jobs in parallel when GPU memory allows.**
#@markdown Results are automatically uploaded to Google Drive after each job completes.
%%time

import subprocess
import os
import zipfile
import time
import gc
import threading
import queue
import re
from datetime import datetime

def get_unique_job_dir(job_name):
    """Return a unique job directory, appending _2, _3, etc. if exists."""
    if not os.path.exists(job_name):
        return job_name
    n = 2
    while os.path.exists(f"{job_name}_{n}"):
        n += 1
    return f"{job_name}_{n}"

def has_existing_output(job_name, search_dir="/content"):
    """Return True if job_name/ contains any .cif or .pdb structure files."""
    job_dir = os.path.join(search_dir, job_name)
    if not os.path.isdir(job_dir):
        return False
    for root, _, files in os.walk(job_dir):
        for f in files:
            if f.endswith(('.cif', '.pdb')):
                return True
    return False

# ============================================================
# VERIFY SETUP
# ============================================================
if 'global_settings' not in globals():
    print("Error: Please run the previous configuration cells first")
    raise ValueError("Configuration not found")

if not global_settings.get('batch_jobs'):
    print("Error: No batch jobs configured")
    print("   Please run Cell 4 to upload and process your CSV")
    raise ValueError("No jobs to process")

config = global_settings['config']
jobs = global_settings['batch_jobs']
processor = global_settings['processor']
drive = global_settings.get('drive')
gdrive_folder_name = global_settings.get('gdrive_folder_name', 'Chai1_Results')


# Check ipSAE_batch availability
ipsae_available = False
try:
    result = subprocess.run(["ipsae-batch", "--help"], capture_output=True, text=True, timeout=10)
    ipsae_available = (result.returncode == 0)
except Exception:
    pass
if config.get('enable_ipsae') and not ipsae_available:
    print("WARNING: ipSAE_batch not installed. Interface analysis will be skipped.")
# ============================================================
# PARALLEL EXECUTION SETTINGS
# ============================================================
enable_parallel = True #@param {type:"boolean"}
#@markdown - **Enable parallel execution**: Launch multiple jobs simultaneously when VRAM allows
#@markdown > **Note**: Jobs are automatically sorted largest-first for optimal calibration. The largest job calibrates VRAM, giving the most accurate per-token rate for parallel scheduling.

vram_limit = 0.85 #@param {type:"slider", min:0.6, max:0.95, step:0.05}
#@markdown - **VRAM limit**: Max fraction of GPU memory to use (0.85 = 85%). Jobs won't launch if this would be exceeded.

# ============================================================
# GPU VERIFICATION + BFLOAT16 COMPATIBILITY CHECK
# ============================================================
print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)
gpu_available = False
total_gpu_vram = 0
gpu_bf16_ok = False

try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        total_gpu_vram = gpu_memory_gb
        cc_major, cc_minor = torch.cuda.get_device_capability(0)
        compute_cap = float(f"{cc_major}.{cc_minor}")
        gpu_bf16_ok = cc_major >= 8

        print(f"GPU: {gpu_name}")
        print(f"   Total Memory: {gpu_memory_gb:.1f} GB")
        print(f"   Compute Capability: {compute_cap}")
        print(f"   bfloat16 Support: {'YES' if gpu_bf16_ok else 'NO'}")
        print(f"   VRAM limit ({vram_limit*100:.0f}%): {gpu_memory_gb * vram_limit:.1f} GB")

        if not gpu_bf16_ok:
            print()
            print("!" * 60)
            print("CRITICAL: GPU DOES NOT SUPPORT bfloat16!")
            print("!" * 60)
            print(f"Your GPU ({gpu_name}) has compute capability {compute_cap}")
            print("Chai-1 REQUIRES bfloat16 support (compute capability >= 8.0)")
            print()
            print("Incompatible GPUs: T4 (7.5), V100 (7.0), P100 (6.0)")
            print("Compatible GPUs: A100 (8.0), L4 (8.9), A10G (8.6), H100 (9.0)")
            print()
            print("To fix: Go to Runtime -> Change runtime type -> Select A100 or L4 GPU")
            print("!" * 60)
            print()
            print("ABORTING: Cannot run Chai-1 on this GPU.")
            raise RuntimeError(f"GPU {gpu_name} (cc {compute_cap}) does not support bfloat16. Chai-1 requires cc >= 8.0.")

        test_tensor = torch.zeros(1).cuda()
        del test_tensor
        torch.cuda.synchronize()
        gpu_available = True

        try:
            result = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                                  capture_output=True, text=True, timeout=2)
            if result.returncode == 0:
                initial_mem = float(result.stdout.strip()) / 1024
                print(f"   Current Usage: {initial_mem:.2f} GB")
                print(f"   GPU monitoring ready")
            else:
                print(f"   nvidia-smi not available, parallel mode disabled")
                enable_parallel = False
        except Exception as e:
            print(f"   nvidia-smi test failed: {e}")
            enable_parallel = False
    else:
        print("WARNING: No GPU detected")
        print("   Predictions will be very slow on CPU")
        enable_parallel = False
except Exception as e:
    if 'bfloat16' in str(e) or 'cc ' in str(e):
        raise  # Re-raise the bfloat16 compatibility error
    print(f"GPU test failed: {e}")
    gpu_available = False
    enable_parallel = False

# Set Chai downloads directory
os.environ['CHAI_DOWNLOADS_DIR'] = '/content/chai_weights'
os.makedirs('/content/chai_weights', exist_ok=True)

# ============================================================
# GPU MONITOR CLASS
# ============================================================
class GPUMonitor:
    """Monitor GPU memory usage using nvidia-smi."""

    def __init__(self):
        self.monitoring = False
        self.peak_memory = 0
        self.current_memory = 0
        self.monitor_thread = None
        self._lock = threading.Lock()
        self.gpu_available = self._test_nvidia_smi()
        self._pid_registry = {}  # shell_pid -> {name, peak_gb}

    def _test_nvidia_smi(self):
        try:
            result = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                                  capture_output=True, text=True, timeout=2)
            return result.returncode == 0
        except:
            return False

    def _get_gpu_memory(self):
        if not self.gpu_available:
            return 0
        try:
            result = subprocess.run(
                ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                capture_output=True, text=True, timeout=2
            )
            if result.returncode == 0:
                return float(result.stdout.strip()) / 1024  # MB to GB
        except:
            pass
        return 0

    def _monitor_loop(self):
        while self.monitoring:
            mem = self._get_gpu_memory()
            # Per-PID VRAM tracking
            job_mem = {}
            with self._lock:
                need_pid = bool(self._pid_registry)
                shells = set(self._pid_registry.keys()) if need_pid else set()
            if need_pid:
                pid_mem = self._get_pid_mem()
                for gpu_pid, gb in pid_mem.items():
                    shell = self._find_job_for_gpu_pid(gpu_pid, shells)
                    if shell is not None:
                        job_mem[shell] = job_mem.get(shell, 0) + gb
            with self._lock:
                self.current_memory = mem
                if mem > self.peak_memory:
                    self.peak_memory = mem
                for spid, cur_gb in job_mem.items():
                    if spid in self._pid_registry:
                        if cur_gb > self._pid_registry[spid]['peak_gb']:
                            self._pid_registry[spid]['peak_gb'] = cur_gb
            time.sleep(0.5)

    def start(self):
        if not self.gpu_available:
            return
        with self._lock:
            self.monitoring = True
            self.peak_memory = 0
            self.current_memory = 0
        self.monitor_thread = threading.Thread(target=self._monitor_loop, daemon=True)
        self.monitor_thread.start()

    def stop(self):
        self.monitoring = False
        if self.monitor_thread:
            self.monitor_thread.join(timeout=2)
        return self.peak_memory

    def get_current(self):
        with self._lock:
            return self.current_memory

    def get_peak(self):
        with self._lock:
            return self.peak_memory

    def reset_peak(self):
        with self._lock:
            self.peak_memory = self.current_memory

    def _get_pid_mem(self):
        """Get per-PID GPU memory {pid: GB}."""
        try:
            r = subprocess.run(
                ['nvidia-smi', '--query-compute-apps=pid,used_gpu_memory', '--format=csv,noheader,nounits'],
                capture_output=True, text=True, timeout=2
            )
            if r.returncode == 0 and r.stdout.strip():
                result = {}
                for ln in r.stdout.strip().split('\n'):
                    parts = ln.strip().split(',')
                    if len(parts) >= 2:
                        try:
                            result[int(parts[0].strip())] = float(parts[1].strip()) / 1024
                        except ValueError:
                            pass
                return result
        except:
            pass
        return {}

    def _find_job_for_gpu_pid(self, gpu_pid, registered_shells):
        """Walk process tree up to find which registered job owns this GPU PID."""
        current = gpu_pid
        visited = set()
        while current > 1 and current not in visited:
            visited.add(current)
            if current in registered_shells:
                return current
            try:
                with open(f'/proc/{current}/status') as f:
                    for line in f:
                        if line.startswith('PPid:'):
                            current = int(line.split(':')[1].strip())
                            break
                    else:
                        break
            except (FileNotFoundError, PermissionError, ValueError):
                break
        return None

    def register_pid(self, shell_pid, job_name):
        """Register a job process PID for per-job VRAM tracking."""
        with self._lock:
            self._pid_registry[shell_pid] = {'name': job_name, 'peak_gb': 0.0}

    def unregister_pid(self, shell_pid):
        """Unregister and return peak VRAM (GB) for a job."""
        with self._lock:
            entry = self._pid_registry.pop(shell_pid, None)
            return entry['peak_gb'] if entry else 0.0

    def get_pid_peak(self, shell_pid):
        """Get peak VRAM (GB) for a specific registered job."""
        with self._lock:
            entry = self._pid_registry.get(shell_pid)
            return entry['peak_gb'] if entry else 0.0


gpu_monitor = GPUMonitor()

def clear_gpu_cache():
    if not gpu_available:
        return
    try:
        import torch
        torch.cuda.empty_cache()
        gc.collect()
    except:
        pass

# ============================================================
# GOOGLE DRIVE HELPERS
# ============================================================
def get_or_create_folder(drive, folder_name, parent_id='root'):
    if not drive:
        return None
    try:
        file_list = drive.ListFile({
            'q': f"'{parent_id}' in parents and title='{folder_name}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
        }).GetList()
        if file_list:
            print(f"Found existing folder: {folder_name}")
            return file_list[0]['id']
        else:
            folder = drive.CreateFile({
                'title': folder_name,
                'mimeType': 'application/vnd.google-apps.folder',
                'parents': [{'id': parent_id}]
            })
            folder.Upload()
            print(f"Created new folder: {folder_name}")
            return folder['id']
    except Exception as e:
        print(f"Error with folder '{folder_name}': {e}")
        return None

def upload_to_gdrive(drive, file_path, folder_id, job_name):
    if not drive or not os.path.exists(file_path):
        return None
    try:
        uploaded_file = drive.CreateFile({
            'title': os.path.basename(file_path),
            'parents': [{'id': folder_id}]
        })
        uploaded_file.SetContentFile(file_path)
        uploaded_file.Upload()
        file_url = f"https://drive.google.com/file/d/{uploaded_file['id']}/view"
        print(f"  Uploaded to Google Drive: {file_url}")
        # Close file handle to prevent ResourceWarning
        if hasattr(uploaded_file, 'content') and uploaded_file.content:
            try:
                uploaded_file.content.close()
            except Exception:
                pass
        return file_url
    except Exception as e:
        print(f"  WARNING: Upload failed: {e}")
        return None

# ============================================================
# UTILITY FUNCTIONS
# ============================================================
def find_output_files(output_dir):
    """Find Chai-1 output files (pred.model_idx_*.cif, scores.model_idx_*.npz, pae.model_idx_*.npz)."""
    cif_files = []
    score_files = []
    pae_files = []
    if os.path.exists(output_dir):
        for root, dirs, files in os.walk(output_dir):
            for f in files:
                if f.startswith('pred.model_idx_') and f.endswith('.cif'):
                    cif_files.append(os.path.join(root, f))
                elif f.startswith('scores.model_idx_') and f.endswith('.npz'):
                    score_files.append(os.path.join(root, f))
                elif f.startswith('pae.model_idx_') and f.endswith('.npz'):
                    pae_files.append(os.path.join(root, f))
    return sorted(cif_files), sorted(score_files), sorted(pae_files)

def parse_chai_progress(line):
    """Check if a line contains Chai-1 progress information."""
    keywords = ['progress', 'step', 'recycl', 'diffus', 'sample',
                'complet', 'saving', 'msa', 'model', 'generat',
                'loading', 'process', 'predict', 'download',
                'trunk', 'denois', 'fold', 'running', 'inference']
    line_lower = line.lower()
    return any(kw in line_lower for kw in keywords)

def count_job_tokens(job, processor):
    """Count tokens for VRAM estimation."""
    total_tokens = 0
    for seq in job.get('sequences', []):
        seq_type = seq.get('type', '').lower()
        if seq_type in ['protein', 'dna', 'rna']:
            total_tokens += len(seq.get('sequence', ''))
        elif seq_type == 'ligand':
            smiles = seq.get('smiles', '')
            original = seq.get('original_code', '')
            # Use atom count from lookup if available
            if original.upper() in processor.ligand_lookup:
                total_tokens += processor.ligand_lookup[original.upper()].get('atom_count', 20)
            elif original.upper() in processor.ion_lookup:
                total_tokens += processor.ion_lookup[original.upper()].get('atom_count', 1)
            elif original.upper() in processor.glycan_lookup:
                total_tokens += processor.glycan_lookup[original.upper()].get('atom_count', 15)
            else:
                # SMILES - estimate conservatively
                total_tokens += 30
    return max(total_tokens, 1)

# ============================================================
# NON-BLOCKING PROCESS OUTPUT READER
# ============================================================

# ============================================================
# MSA FOLDER LOOKUP + A3M TO PQT CONVERTER (Chai-1 specific)
# ============================================================
def find_precomputed_msas(job, msa_folder):
    """Look up pre-computed MSA files for ALL protein chains in a job.

    All-or-nothing: returns MSA paths only if EVERY protein chain has a
    pre-computed MSA file. If any chain is missing, returns empty dict
    and the job falls back to online MSA search entirely.
    """
    if not msa_folder or not os.path.isdir(msa_folder):
        return {}

    job_name = job['name']
    found = {}
    missing = []

    seen = set()
    protein_chains = []
    for seq in job.get('sequences', []):
        if seq.get('type', '').lower() != 'protein':
            continue
        chain_name = seq.get('name', '')
        if not chain_name or chain_name in seen:
            continue
        seen.add(chain_name)
        protein_chains.append(chain_name)

    if not protein_chains:
        return {}

    available_files = {}
    for fname in os.listdir(msa_folder):
        if fname.endswith('_paired.a3m'):
            normalized = fname.lower().replace('-', '_')
            available_files[normalized] = fname

    for chain_name in protein_chains:
        expected = f"{job_name}_{chain_name}_paired.a3m"
        exact_path = os.path.join(msa_folder, expected)
        if os.path.isfile(exact_path):
            found[chain_name] = exact_path
        else:
            norm_key = expected.lower().replace('-', '_')
            if norm_key in available_files:
                found[chain_name] = os.path.join(msa_folder, available_files[norm_key])
            else:
                missing.append(chain_name)

    if missing:
        print(f"   Pre-computed MSAs missing for: {missing} -- using online MSA for all chains")
        return {}

    print(f"   Pre-computed MSAs found for all {len(found)} chain(s): {list(found.keys())}")
    return found


def convert_a3m_to_pqt(a3m_path, output_dir, pqt_filename):
    """Convert .a3m file to Chai-1 .aligned.pqt format.

    Chai-1 expects .aligned.pqt (Parquet) with columns:
    - sequence: aligned sequence string
    - source_database: "query" for first row, "uniref90" for homologs
    - pairing_key: str for cross-chain pairing (from OX= TaxID), "" for query
    - comment: empty string

    pqt_filename should be the SHA-256 hash of the protein sequence,
    since Chai-1 looks up MSA files by sequence hash.
    """
    import re
    import pandas as pd_conv

    sequences = []
    source_dbs = []
    pairing_keys = []
    comments = []

    with open(a3m_path) as f:
        current_seq_lines = []
        current_taxid = ""
        in_entry = False
        is_first = True

        for line in f:
            line = line.rstrip()
            if line.startswith('>'):
                if in_entry and current_seq_lines:
                    sequences.append(''.join(current_seq_lines))
                    if is_first:
                        source_dbs.append("query")
                        pairing_keys.append("")
                        is_first = False
                    else:
                        source_dbs.append("uniref90")
                        pairing_keys.append(current_taxid)
                    comments.append("")
                current_seq_lines = []
                ox_match = re.search(r'OX=(\d+)', line)
                current_taxid = ox_match.group(1) if ox_match else ""
                in_entry = True
            elif in_entry:
                current_seq_lines.append(line)

        if in_entry and current_seq_lines:
            sequences.append(''.join(current_seq_lines))
            if is_first:
                source_dbs.append("query")
                pairing_keys.append("")
            else:
                source_dbs.append("uniref90")
                pairing_keys.append(current_taxid)
            comments.append("")

    if not sequences:
        return None

    df = pd_conv.DataFrame({
        'sequence': sequences,
        'source_database': source_dbs,
        'pairing_key': pairing_keys,
        'comment': comments,
    })

    pqt_path = os.path.join(output_dir, f"{pqt_filename}.aligned.pqt")
    df.to_parquet(pqt_path, index=False)
    return pqt_path


def prepare_chai_msa_directory(job, msa_folder, job_dir):
    """Prepare MSA directory for Chai-1 from pre-computed .a3m files.

    Returns path to msa directory if ALL protein chains have MSAs, else None.
    Chai-1 is all-or-nothing for --msa-directory.
    """
    if not msa_folder:
        return None

    precomp = find_precomputed_msas(job, msa_folder)
    if not precomp:
        return None

    protein_chains = [s['name'] for s in job.get('sequences', [])
                      if s.get('type', '').lower() == 'protein' and s.get('name')]
    missing = [c for c in protein_chains if c not in precomp]
    if missing:
        print(f"   Incomplete MSA coverage (missing: {missing}) - using online MSA")
        return None

    # Build chain_name -> sequence mapping for SHA-256 filenames
    # Chai-1 looks up MSA files by sha256(sequence), not by chain name
    import hashlib as _hl
    name_to_seq = {}
    for s in job.get('sequences', []):
        if s.get('type', '').lower() == 'protein' and s.get('name') and s.get('sequence'):
            name_to_seq[s['name']] = s['sequence']

    msa_dir = os.path.join(job_dir, "precomputed_msas")
    os.makedirs(msa_dir, exist_ok=True)

    for chain_name, a3m_path in precomp.items():
        seq = name_to_seq.get(chain_name, '')
        if not seq:
            print(f"   No sequence found for {chain_name} - using online MSA")
            return None
        seq_hash = _hl.sha256(seq.encode()).hexdigest()
        pqt_path = convert_a3m_to_pqt(a3m_path, msa_dir, seq_hash)
        if not pqt_path:
            print(f"   Failed to convert {chain_name}.a3m to .pqt - using online MSA")
            return None

    print(f"   Converted {len(precomp)} MSAs to .aligned.pqt format")
    return msa_dir

# Known harmless warnings to suppress from subprocess output
_SUPPRESS_PATTERNS = [
    'FutureWarning',
    'Non-SM100f kernel expects bias to be float32',
    'builtin type swigvarlink has no __module__',
    'builtin type SwigPyPacked has no __module__',
    'builtin type SwigPyObject has no __module__',
    'datetime.datetime.utcnow() is deprecated',
]

def _is_suppressed_warning(line):
    """Check if a line matches known harmless warning patterns."""
    return any(pat in line for pat in _SUPPRESS_PATTERNS)

def _reader_thread(pipe, output_queue):
    """Read lines from a pipe into a thread-safe queue."""
    try:
        for line in iter(pipe.readline, ''):
            output_queue.put(line.rstrip('\n'))
    except:
        pass
    finally:
        try:
            pipe.close()
        except:
            pass

# ============================================================
# BUILD CHAI-1 COMMAND
# ============================================================
def build_chai_cmd(fasta_file, output_dir, current_config):
    """Build the chai-lab fold command."""
    cmd_parts = [
        "chai-lab", "fold",
        fasta_file,
        output_dir,
        "--num-trunk-recycles", str(current_config['num_trunk_recycles']),
        "--num-diffn-timesteps", str(current_config['num_diffn_timesteps']),
        "--seed", str(current_config['seed']),
    ]

    if current_config.get('_msa_directory'):
        cmd_parts.extend(["--msa-directory", current_config['_msa_directory']])
    elif current_config.get('use_msa_server'):
        cmd_parts.append("--use-msa-server")
        if current_config.get('msa_server_url') and current_config['msa_server_url'] != "https://api.colabfold.com":
            cmd_parts.extend(["--msa-server-url", current_config['msa_server_url']])

    num_diffn = current_config.get('num_diffn_samples', 5)
    cmd_parts.extend(["--num-diffn-samples", str(num_diffn)])

    num_trunk = current_config.get('num_trunk_samples', 1)
    cmd_parts.extend(["--num-trunk-samples", str(num_trunk)])

    if not current_config.get('use_esm_embeddings', True):
        cmd_parts.append("--no-use-esm-embeddings")

    if not current_config.get('low_memory', True):
        cmd_parts.append("--no-low-memory")

    return " ".join(cmd_parts)


# ============================================================
# ipSAE INTERFACE ANALYSIS
# ============================================================
def run_ipsae_analysis(job_dir, job_name):
    """Run ipSAE per-job analysis. Returns True on success, False on failure."""
    if not ipsae_available or not config.get('enable_ipsae'):
        return False

    ipsae_output = os.path.join(job_dir, "ipsae_analysis")
    os.makedirs(ipsae_output, exist_ok=True)

    cmd_parts = [
        "ipsae-batch", job_dir,
        "--backend", "chai1",
        "--output_dir", ipsae_output,
        "--pae_cutoff", str(config.get('ipsae_pae_cutoff', 10.0)),
        "--dist_cutoff", str(config.get('ipsae_dist_cutoff', 10.0)),
        "--workers", "1",
        "--cores", "1",
    ]
    if config.get('ipsae_png'):
        cmd_parts.append("--png")
    if config.get('ipsae_pdf'):
        cmd_parts.append("--pdf")
    if config.get('ipsae_per_residue'):
        cmd_parts.append("--per_residue")
    if config.get('ipsae_per_contact'):
        cmd_parts.append("--per_contact")
    if config.get('ipsae_select_residues'):
        cmd_parts.extend(["--select-residues", config['ipsae_select_residues']])

    print(f"   Running ipSAE analysis...")
    try:
        result = subprocess.run(cmd_parts, capture_output=True, text=True, timeout=300)
        if result.returncode == 0:
            n_files = sum(1 for f in os.listdir(ipsae_output) if not f.startswith('.'))
            print(f"   ipSAE complete: {n_files} output files")
            return True
        else:
            print(f"   ipSAE failed (rc={result.returncode}): {result.stderr[:200]}")
            return False
    except subprocess.TimeoutExpired:
        print(f"   ipSAE timed out (300s)")
        return False
    except Exception as e:
        print(f"   ipSAE error: {e}")
        return False

# ============================================================
# PROCESS COMPLETED JOB (zip + upload + collect results)
# ============================================================
def process_completed_job(job_name, job_dir, output_dir, fasta_file, job_time, peak_gpu, drive, gdrive_folder_id):
    """Find outputs, zip, upload to GDrive. Returns result dict."""
    cif_files, score_files, pae_files = find_output_files(output_dir)

    if not cif_files:
        print(f"\nNo structure files (.cif) found for {job_name}")
        # Show directory structure for debugging
        print(f"   Directory structure of {job_dir}:")
        for root, dirs, files in os.walk(job_dir):
            level = root.replace(job_dir, '').count(os.sep)
            indent = '   ' * (level + 1)
            print(f"{indent}{os.path.basename(root)}/")
            for file in files[:10]:
                print(f"{indent}  {file}")
            if len(files) > 10:
                print(f"{indent}  ... and {len(files) - 10} more files")
        return {
            'success': False, 'name': job_name,
            'error': 'No structure files generated',
            'time': job_time, 'gpu_peak': peak_gpu
        }

    print(f"   {len(cif_files)} structure files generated")
    for f in sorted(cif_files)[:3]:
        print(f"      {f}")
    if len(cif_files) > 3:
        print(f"      ... and {len(cif_files) - 3} more")
    if score_files:
        print(f"   {len(score_files)} score files generated")
    if pae_files:
        print(f"   {len(pae_files)} PAE files generated")

    # --- ipSAE interface analysis (before zipping) ---
    ipsae_ran = False
    try:
        ipsae_ran = run_ipsae_analysis(job_dir, job_name)
    except Exception as e:
        print(f"   ipSAE error (non-fatal): {e}")

    # Create ZIP
    zip_filename = f"{job_name}_results.zip"
    zip_path = os.path.join(job_dir, zip_filename)
    try:
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            # Add all output files
            for root, dirs, files in os.walk(output_dir):
                for file in files:
                    file_path = os.path.join(root, file)
                    arcname = os.path.relpath(file_path, job_dir)
                    zipf.write(file_path, arcname)
            # Add the FASTA input file
            if os.path.exists(fasta_file):
                zipf.write(fasta_file, os.path.basename(fasta_file))
                # Add environment log for traceability
                if os.path.isfile("/content/environment.txt"):
                    zipf.write("/content/environment.txt", "environment.txt")
            # Add ipSAE analysis files
            ipsae_dir = os.path.join(job_dir, "ipsae_analysis")
            if os.path.isdir(ipsae_dir):
                for root, dirs, files_list in os.walk(ipsae_dir):
                    for f in files_list:
                        fp = os.path.join(root, f)
                        arcname = os.path.join("ipsae_analysis", os.path.relpath(fp, ipsae_dir))
                        zipf.write(fp, arcname)
        print(f"   Created: {zip_filename}")
    except Exception as e:
        print(f"   Failed to create ZIP: {e}")

    # Upload to GDrive
    file_url = None
    if drive and gdrive_folder_id:
        file_url = upload_to_gdrive(drive, zip_path, gdrive_folder_id, job_name)
    else:
        print(f"   Saved locally: {zip_path}")

    return {
        'success': True, 'name': job_name,
        'structures': len(cif_files),
        'scores': len(score_files),
        'time': job_time, 'url': file_url,
        'gpu_peak': peak_gpu,
        'ipsae_ran': ipsae_ran,
    }

# ============================================================
# SEQUENTIAL RUNNER
# ============================================================
def run_single_prediction_sequential(job, job_idx, total_jobs):
    """Run a single Chai-1 prediction sequentially with GPU monitoring."""
    job_start = time.time()

    print("\n" + "=" * 60)
    print(f"Job {job_idx}/{total_jobs}: {job['name']}")
    print("=" * 60)

    if config.get('skip_existing', False) and has_existing_output(job['name']):
        print(f"  \u23ed Skipping {job['name']}: output already exists")
        return {'name': job['name'], 'skipped': True, 'success': True,
                'structures': 0, 'time': 0, 'gpu_peak': 0,
                'tokens': count_job_tokens(job, processor)}

    gpu_monitor.start()
    time.sleep(0.5)
    initial_mem = gpu_monitor.get_current()
    print(f"GPU Memory (start): {initial_mem:.2f} GB")

    job_name = job['name']
    job_dir = get_unique_job_dir(job_name)
    output_dir = os.path.join(job_dir, "output")
    os.makedirs(job_dir, exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)

    # Generate FASTA
    fasta_content = processor._generate_fasta(job)
    fasta_file = os.path.join(job_dir, f"{job_name}.fasta")
    with open(fasta_file, 'w') as f:
        f.write(fasta_content)

    print(f"   FASTA file: {fasta_file}")
    print(f"   Sequences: {len(job['sequences'])}")

    # Build command
    # Prepare pre-computed MSA directory if available
    msa_dir = prepare_chai_msa_directory(job, global_settings.get('msa_folder', ''), job_dir)
    run_config = dict(config)
    if msa_dir:
        run_config['_msa_directory'] = msa_dir
    cmd = build_chai_cmd(fasta_file, output_dir, run_config)
    print(f"Command: {cmd}")
    print(f"\nRunning Chai-1 prediction...")

    # Phase tracking
    phase_times = {}
    current_phase = "initialization"
    phase_start = time.time()

    try:
        process = subprocess.Popen(
            cmd, shell=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, universal_newlines=True,
            env={**os.environ, 'CHAI_DOWNLOADS_DIR': '/content/chai_weights'}
        )

        # Non-blocking reader
        output_q = queue.Queue()
        reader_t = threading.Thread(target=_reader_thread, args=(process.stdout, output_q), daemon=True)
        reader_t.start()

        print("\nProgress:")
        while process.poll() is None:
            while not output_q.empty():
                try:
                    line = output_q.get_nowait()
                except queue.Empty:
                    break
                if not line:
                    continue

                # Track phases
                ll = line.lower()
                if 'msa' in ll and current_phase != "msa":
                    phase_times[current_phase] = time.time() - phase_start
                    current_phase = "msa"; phase_start = time.time()
                elif ('recycl' in ll or 'trunk' in ll) and current_phase != "recycling":
                    phase_times[current_phase] = time.time() - phase_start
                    current_phase = "recycling"; phase_start = time.time()
                elif ('diffus' in ll or 'denois' in ll) and current_phase != "diffusion":
                    phase_times[current_phase] = time.time() - phase_start
                    current_phase = "diffusion"; phase_start = time.time()
                elif 'saving' in ll and current_phase != "saving":
                    phase_times[current_phase] = time.time() - phase_start
                    current_phase = "saving"; phase_start = time.time()
                elif 'download' in ll and current_phase != "download_weights":
                    phase_times[current_phase] = time.time() - phase_start
                    current_phase = "download_weights"; phase_start = time.time()

                # Print progress
                if parse_chai_progress(line):
                    cur_gpu = gpu_monitor.get_current()
                    if cur_gpu > 1.0:
                        print(f"   {line[:100]} [GPU: {cur_gpu:.1f} GB]")
                    else:
                        print(f"   {line[:100]}")
                elif ('error' in line.lower() or 'warning' in line.lower()) and not _is_suppressed_warning(line):
                    print(f"   WARNING: {line[:100]}")

            time.sleep(0.5)

        # Drain remaining output
        while not output_q.empty():
            try:
                line = output_q.get_nowait()
                if line and (parse_chai_progress(line) or 'error' in line.lower()):
                    print(f"   {line[:100]}")
            except queue.Empty:
                break

        phase_times[current_phase] = time.time() - phase_start
        return_code = process.wait(timeout=300)
        peak_gpu_memory = gpu_monitor.stop()

        print(f"\nGPU Memory: Peak={peak_gpu_memory:.2f} GB, Final={gpu_monitor.get_current():.2f} GB")

        if return_code != 0:
            print(f"\nPrediction failed (return code: {return_code})")
            # Show directory structure for debugging
            print(f"   Directory structure of {job_dir}:")
            for root, dirs, files in os.walk(job_dir):
                level = root.replace(job_dir, '').count(os.sep)
                indent = '   ' * (level + 1)
                print(f"{indent}{os.path.basename(root)}/")
                for file in files[:10]:
                    print(f"{indent}  {file}")
            return {
                'success': False, 'name': job_name,
                'error': f"Process exited with code {return_code}",
                'time': time.time() - job_start, 'gpu_peak': peak_gpu_memory,
                'tokens': count_job_tokens(job, processor)
            }

    except subprocess.TimeoutExpired:
        process.kill()
        peak_gpu_memory = gpu_monitor.stop()
        return {
            'success': False, 'name': job_name, 'error': "Timeout",
            'time': time.time() - job_start, 'gpu_peak': peak_gpu_memory,
            'tokens': count_job_tokens(job, processor)
        }
    except Exception as e:
        peak_gpu_memory = gpu_monitor.stop()
        return {
            'success': False, 'name': job_name, 'error': str(e),
            'time': time.time() - job_start, 'gpu_peak': peak_gpu_memory,
            'tokens': count_job_tokens(job, processor)
        }

    # Print timing
    if phase_times:
        total_phase_time = sum(phase_times.values())
        print(f"\nTiming breakdown:")
        for phase, duration in phase_times.items():
            if duration > 0.1:
                pct = (duration / total_phase_time) * 100 if total_phase_time > 0 else 0
                print(f"   - {phase.capitalize()}: {duration:.1f}s ({pct:.0f}%)")

    job_time = time.time() - job_start
    result = process_completed_job(job_name, job_dir, output_dir, fasta_file, job_time, peak_gpu_memory, drive, gdrive_folder_id)
    if 'tokens' not in result:
        result['tokens'] = count_job_tokens(job, processor)
    print(f"\nTotal: {job_time:.1f}s ({job_time/60:.1f} min)")
    return result

# ============================================================
# SETUP GOOGLE DRIVE
# ============================================================
gdrive_folder_id = None
if drive:
    print("\n" + "=" * 60)
    print("SETTING UP GOOGLE DRIVE")
    print("=" * 60)
    gdrive_folder_id = get_or_create_folder(drive, gdrive_folder_name)
    if gdrive_folder_id:
        print(f"Results will be uploaded to: {gdrive_folder_name}")
    else:
        print("Could not setup Drive folder - results will be saved locally only")
        drive = None
else:
    print("\n" + "=" * 60)
    print("NO GOOGLE DRIVE - LOCAL ONLY")
    print("=" * 60)
    print("Google Drive not configured in Cell 3")

# ============================================================
# PRINT BATCH INFO
# ============================================================
print("\n" + "=" * 60)
print(f"STARTING BATCH PROCESSING: {len(jobs)} JOBS")
print("=" * 60)
print(f"Configuration:")
print(f"  - Trunk recycles: {config['num_trunk_recycles']}")
print(f"  - Diffusion timesteps: {config['num_diffn_timesteps']}")
print(f"  - Seed: {config['seed']}")
print(f"  - MSA server: {config['use_msa_server']}")
print(f"  - Parallel mode: {enable_parallel}")
if enable_parallel and not global_settings.get('msa_folder') and config.get('use_msa_server', True):
    print(f"  - MSA stagger: 90s between launches (ColabFold rate limiting)")
print(f"\nStarted: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Sort jobs largest-first for optimal calibration accuracy
jobs = sorted(jobs, key=lambda j: count_job_tokens(j, processor), reverse=True)
print(f"Jobs sorted by size (largest first):")
for _j in jobs:
    print(f"  {_j['name']}: {count_job_tokens(_j, processor)} tokens")

# ============================================================
# SINGLE JOB SHORTCUT
# ============================================================
if len(jobs) == 1:
    enable_parallel = False

# ============================================================
# MAIN EXECUTION
# ============================================================
batch_start_time = time.time()
completed_jobs = []
failed_jobs = []

if not enable_parallel or not gpu_available:
    # --------------------------------------------------------
    # SEQUENTIAL MODE
    # --------------------------------------------------------
    if not enable_parallel:
        print(f"\nRunning in SEQUENTIAL mode (parallel disabled)")
    else:
        print(f"\nRunning in SEQUENTIAL mode (no GPU monitoring)")

    for job_idx, job in enumerate(jobs, 1):
        result = run_single_prediction_sequential(job, job_idx, len(jobs))

        if result['success']:
            completed_jobs.append(result)
        else:
            failed_jobs.append(result)

        clear_gpu_cache()
        time.sleep(1)

else:
    # --------------------------------------------------------
    # CALIBRATION-BASED PARALLEL MODE
    # --------------------------------------------------------
    print(f"\n" + "=" * 60)
    print(f"CALIBRATION-BASED PARALLEL MODE")
    print("=" * 60)
    print(f"Phase 1: Run job 1 alone to measure actual VRAM usage")
    print(f"Phase 2: Use measurement to estimate remaining jobs")
    print(f"Phase 3: Launch jobs in parallel when VRAM allows")
    print("=" * 60)

    # --- PHASE 1: CALIBRATION (try jobs one by one until one succeeds) ---
    print(f"\n{chr(0x1f4cf)} PHASE 1: CALIBRATION")
    print(f"Running jobs one at a time until one succeeds to measure VRAM...")

    cal_result = None
    cal_peak_vram = 0
    cal_tokens = 0
    calibration_idx = -1

    skipped_in_cal = 0
    for cal_idx, cal_job in enumerate(jobs):
        cal_job_name = cal_job['name']

        # Skip already-completed jobs -- they give no VRAM data
        if config.get('skip_existing', False) and has_existing_output(cal_job_name):
            print(f"  \u23ed Skipping {cal_job_name} (already done)")
            completed_jobs.append({'name': cal_job_name, 'skipped': True, 'success': True,
                                   'structures': 0, 'time': 0, 'gpu_peak': 0,
                                   'tokens': count_job_tokens(cal_job, processor)})
            skipped_in_cal += 1
            continue

        cal_tokens = count_job_tokens(cal_job, processor)
        print(f"\n   Calibration attempt {cal_idx + 1}/{len(jobs)}: {cal_job['name']} ({cal_tokens} tokens)")

        cal_result = run_single_prediction_sequential(cal_job, cal_idx + 1, len(jobs))

        if cal_result['success']:
            completed_jobs.append(cal_result)
            calibration_idx = cal_idx
            cal_peak_vram = cal_result.get('gpu_peak', 0)
            print(f"\n   {chr(0x2705)} Calibration succeeded on job '{cal_job['name']}' -- peak VRAM: {cal_peak_vram:.2f} GB")
            break
        else:
            failed_jobs.append(cal_result)
            print(f"   {chr(0x26a0)}  Job '{cal_job['name']}' failed -- trying next job for calibration...")
            clear_gpu_cache()
            time.sleep(2)
            continue

    remaining_jobs = jobs[calibration_idx + 1:] if calibration_idx >= 0 else []

    if calibration_idx < 0:
        if skipped_in_cal == len(jobs):
            print(f"\n{chr(0x2705)} All {len(jobs)} jobs already completed. Nothing to run.")
        else:
            print(f"\n{chr(0x274c)} ALL {len(jobs)} jobs failed. No successful calibration possible.")
    elif not remaining_jobs:
        print("\nOnly one job - calibration complete, nothing more to run.")
    elif cal_peak_vram <= 0:
        # No VRAM data - fall back to sequential
        print(f"\nCould not measure VRAM during calibration. Falling back to sequential.")
        for job_idx, job in enumerate(remaining_jobs, calibration_idx + 2):
            result = run_single_prediction_sequential(job, job_idx, len(jobs))
            if result['success']:
                completed_jobs.append(result)
            else:
                failed_jobs.append(result)
            clear_gpu_cache()
            time.sleep(1)
    elif cal_peak_vram > total_gpu_vram * vram_limit:
        # Single job already uses most of the GPU - sequential only
        print(f"\nCalibration job used {cal_peak_vram:.1f} GB " +
              f"(>{total_gpu_vram * vram_limit:.1f} GB safe limit)")
        print(f"   Single job already fills GPU. Running remaining {len(remaining_jobs)} jobs sequentially.")

        for job_idx, job in enumerate(remaining_jobs, calibration_idx + 2):
            result = run_single_prediction_sequential(job, job_idx, len(jobs))
            if result['success']:
                completed_jobs.append(result)
            else:
                failed_jobs.append(result)
            clear_gpu_cache()
            time.sleep(1)
    else:
        # --- PHASE 2: ESTIMATE REMAINING JOBS ---
        vram_per_token = cal_peak_vram / cal_tokens
        model_overhead = cal_peak_vram * 0.3  # minimum floor

        print(f"\nPHASE 2: VRAM ESTIMATION")
        print(f"   Calibration: {cal_peak_vram:.2f} GB peak for {cal_tokens} tokens")
        print(f"   Rate: {vram_per_token:.4f} GB/token")
        print(f"   Model overhead floor: {model_overhead:.2f} GB")
        print(f"   Available for parallel: {total_gpu_vram * vram_limit:.1f} GB")

        job_estimates = []
        for job in remaining_jobs:
            tokens = count_job_tokens(job, processor)
            estimated_vram = max(
                vram_per_token * tokens * 1.15,  # 15% safety margin
                model_overhead                    # minimum floor
            )
            job_estimates.append({
                'job': job,
                'tokens': tokens,
                'estimated_vram': estimated_vram
            })
            print(f"   {job['name']}: {tokens} tokens -> ~{estimated_vram:.1f} GB estimated")

        # --- PHASE 3: PARALLEL EXECUTION ---
        print(f"\nPHASE 3: PARALLEL EXECUTION")
        safe_limit = total_gpu_vram * vram_limit
        emergency = total_gpu_vram * vram_limit

        # State tracking
        pending_estimates = list(enumerate(job_estimates, calibration_idx + 2))  # (job_number, estimate)
        running_procs = []  # list of dicts with process info
        emergency_triggered = False

        gpu_monitor.start()

        def try_launch_next():
            """Try to launch the next pending job if VRAM allows."""
            if not pending_estimates:
                return False

            current_vram = gpu_monitor.get_current()
            # Use HIGHER of actual GPU or estimated running (nvidia-smi lags behind)
            estimated_running = sum(p['estimated_vram'] for p in running_procs)
            effective_vram = max(current_vram, estimated_running)
            job_number, est = pending_estimates[0]
            needed = est['estimated_vram']

            if effective_vram + needed < safe_limit:
                pending_estimates.pop(0)
                job = est['job']
                job_name = job['name']

                if config.get('skip_existing', False) and has_existing_output(job_name):
                    print(f"  \u23ed Skipping {job_name}: output already exists")
                    completed_jobs.append({'name': job_name, 'skipped': True, 'success': True,
                                          'structures': 0, 'time': 0, 'gpu_peak': 0,
                                          'tokens': est['tokens']})
                    return False

                job_dir = get_unique_job_dir(job_name)
                output_dir = os.path.join(job_dir, "output")
                os.makedirs(job_dir, exist_ok=True)
                os.makedirs(output_dir, exist_ok=True)

                fasta_content = processor._generate_fasta(job)
                fasta_file = os.path.join(job_dir, f"{job_name}.fasta")
                with open(fasta_file, 'w') as f:
                    f.write(fasta_content)

                # Prepare pre-computed MSA directory if available
                msa_dir_p = prepare_chai_msa_directory(job, global_settings.get('msa_folder', ''), job_dir)
                run_config_p = dict(config)
                if msa_dir_p:
                    run_config_p['_msa_directory'] = msa_dir_p
                cmd = build_chai_cmd(fasta_file, output_dir, run_config_p)

                try:
                    proc = subprocess.Popen(
                        cmd, shell=True,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, universal_newlines=True,
                        env={**os.environ, 'CHAI_DOWNLOADS_DIR': '/content/chai_weights'}
                    )
                except Exception as e:
                    print(f"Failed to launch {job_name}: {e}")
                    failed_jobs.append({
                        'success': False, 'name': job_name,
                        'error': f'Launch failed: {e}', 'time': 0, 'gpu_peak': 0
                    })
                    return try_launch_next()

                # Non-blocking reader
                out_q = queue.Queue()
                reader = threading.Thread(target=_reader_thread, args=(proc.stdout, out_q), daemon=True)
                reader.start()

                info = {
                    'process': proc, 'job': job, 'job_name': job_name,
                    'job_dir': job_dir, 'output_dir': output_dir,
                    'fasta_file': fasta_file,
                    'job_number': job_number, 'tokens': est['tokens'],
                    'estimated_vram': needed, 'start_time': time.time(),
                    'output_queue': out_q
                }
                running_procs.append(info)
                gpu_monitor.register_pid(proc.pid, job_name)

                print(f"\nLaunched job {job_number}/{len(jobs)}: {job_name} "
                      f"(est. {needed:.1f} GB, GPU: {effective_vram:.1f}/{safe_limit:.1f} GB)")

                # Stagger launches when using ColabFold MSA server to avoid rate limiting
                if not global_settings.get('msa_folder') and config.get('use_msa_server', True):
                    print("   Staggering launch (90s) to avoid ColabFold rate limiting...")
                    time.sleep(90)
                else:
                    time.sleep(2)  # brief pause for VRAM allocation
                return True
            return False

        # Launch initial jobs
        print(f"\nLaunching parallel jobs...")
        while try_launch_next():
            pass

        if running_procs:
            print(f"\n{len(running_procs)} job(s) running, {len(pending_estimates)} pending")
        else:
            print(f"\nCould not launch any parallel jobs")

        # Main monitoring loop
        while running_procs or pending_estimates:
            for info in list(running_procs):
                # Drain output queue
                while not info['output_queue'].empty():
                    try:
                        line = info['output_queue'].get_nowait()
                        if not line:
                            continue
                        if parse_chai_progress(line):
                            cur = gpu_monitor.get_current()
                            print(f"   [{info['job_name']}] {line[:80]} [GPU: {cur:.1f} GB]")
                        elif ('error' in line.lower() or 'warning' in line.lower()) and not _is_suppressed_warning(line):
                            print(f"   WARNING [{info['job_name']}] {line[:80]}")
                    except queue.Empty:
                        break

                # Check completion
                rc = info['process'].poll()
                if rc is not None:
                    running_procs.remove(info)
                    job_time = time.time() - info['start_time']
                    pid_peak = gpu_monitor.unregister_pid(info['process'].pid)

                    # Final drain: capture any lines buffered between last poll and exit
                    while not info['output_queue'].empty():
                        try:
                            info['output_queue'].get_nowait()
                        except queue.Empty:
                            break

                    try:
                        if rc == 0:
                            print(f"\n{chr(0x2705)} Completed: {info['job_name']} ({job_time:.1f}s, VRAM: {pid_peak:.1f} GB)")
                            result = process_completed_job(
                                info['job_name'], info['job_dir'], info['output_dir'],
                                info['fasta_file'],
                                job_time, pid_peak, drive, gdrive_folder_id
                            )
                            result['tokens'] = info['tokens']
                            if result['success']:
                                completed_jobs.append(result)
                            else:
                                failed_jobs.append(result)
                        else:
                            print(f"\n{chr(0x274c)} Failed: {info['job_name']} (exit code {rc}, {job_time:.1f}s, VRAM: {pid_peak:.1f} GB)")
                            failed_jobs.append({
                                'success': False, 'name': info['job_name'],
                                'error': f'Exit code {rc}',
                                'time': job_time, 'gpu_peak': pid_peak
                            })
                    except Exception as e:
                        print(f"  ERROR processing {info['job_name']}: {e}")
                        failed_jobs.append({
                            'success': False, 'name': info['job_name'],
                            'error': str(e), 'time': job_time, 'gpu_peak': pid_peak
                        })

                    # Try launching next
                    clear_gpu_cache()
                    time.sleep(1)
                    gpu_monitor.reset_peak()

                    if not emergency_triggered:
                        while try_launch_next():
                            pass
                        if running_procs or pending_estimates:
                            print(f"   Status: {len(running_procs)} running, "
                                  f"{len(pending_estimates)} pending, "
                                  f"{len(completed_jobs)} done, {len(failed_jobs)} failed")

            # Emergency VRAM check
            current_vram = gpu_monitor.get_current()
            if current_vram > emergency and not emergency_triggered:
                emergency_triggered = True
                print(f"\nEMERGENCY: GPU VRAM at {current_vram:.1f}/{total_gpu_vram:.1f} GB")
                print(f"   Stopping new launches. Waiting for {len(running_procs)} running jobs...")

            time.sleep(2)

        peak_vram = gpu_monitor.stop()

        # If there are remaining jobs after emergency, run sequentially
        if pending_estimates:
            print(f"\nRunning {len(pending_estimates)} remaining jobs sequentially after emergency...")
            for job_number, est in pending_estimates:
                result = run_single_prediction_sequential(est['job'], job_number, len(jobs))
                if result['success']:
                    completed_jobs.append(result)
                else:
                    failed_jobs.append(result)
                clear_gpu_cache()
                time.sleep(1)


# ============================================================
# FINAL ipSAE BATCH ANALYSIS
# ============================================================
ipsae_batch_url = None
if (ipsae_available and config.get('enable_ipsae')
        and config.get('ipsae_batch_analysis') and len(completed_jobs) > 1):
    print("\n" + "=" * 60)
    print("FINAL ipSAE BATCH ANALYSIS")
    print("=" * 60)

    ipsae_batch_output = "/content/ipsae_batch_results"
    os.makedirs(ipsae_batch_output, exist_ok=True)

    cmd_parts = [
        "ipsae-batch", "/content",
        "--backend", "chai1",
        "--output_dir", ipsae_batch_output,
        "--pae_cutoff", str(config.get('ipsae_pae_cutoff', 10.0)),
        "--dist_cutoff", str(config.get('ipsae_dist_cutoff', 10.0)),
        "--cores", str(os.cpu_count() or 2),
    ]
    if config.get('ipsae_per_residue'):
        cmd_parts.append("--per_residue")
    if config.get('ipsae_per_contact'):
        cmd_parts.append("--per_contact")
    if config.get('ipsae_select_residues'):
        cmd_parts.extend(["--select-residues", config['ipsae_select_residues']])

    print(f"Analyzing {len(completed_jobs)} completed jobs...")
    try:
        result = subprocess.run(cmd_parts, capture_output=True, text=True, timeout=600)
        if result.returncode == 0:
            batch_files = [f for f in os.listdir(ipsae_batch_output) if not f.startswith('.')]
            print(f"Batch analysis complete: {len(batch_files)} files")
            for bf in sorted(batch_files):
                size = os.path.getsize(os.path.join(ipsae_batch_output, bf))
                print(f"   {bf} ({size//1024}KB)")

            # Upload to GDrive
            if drive and gdrive_folder_id:
                batch_zip = "/content/ipsae_batch_results.zip"
                with zipfile.ZipFile(batch_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
                    for root, dirs, files_list in os.walk(ipsae_batch_output):
                        for f in files_list:
                            fp = os.path.join(root, f)
                            arcname = os.path.join("ipsae_batch_results", os.path.relpath(fp, ipsae_batch_output))
                            zipf.write(fp, arcname)
                ipsae_batch_url = upload_to_gdrive(drive, batch_zip, gdrive_folder_id, "ipsae_batch")
                print(f"Uploaded: {ipsae_batch_url}")
            else:
                print(f"Saved locally: {ipsae_batch_output}/")
        else:
            print(f"Batch analysis failed (rc={result.returncode})")
            if result.stderr:
                print(f"   {result.stderr[:300]}")
    except subprocess.TimeoutExpired:
        print("Batch analysis timed out (600s)")
    except Exception as e:
        print(f"Batch analysis error: {e}")

# ============================================================
# FINAL SUMMARY
# ============================================================
total_time = time.time() - batch_start_time

print("\n" + "=" * 60)
print("BATCH PROCESSING COMPLETE")
print("=" * 60)
print(f"Ended: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total duration: {total_time/60:.1f} minutes ({total_time/3600:.2f} hours)")
print(f"\nCompleted: {len(completed_jobs)}/{len(jobs)} jobs")
print(f"Failed: {len(failed_jobs)}/{len(jobs)} jobs")

if completed_jobs:
    print(f"\nSuccessful jobs:")
    for job in completed_jobs:
        if job.get('skipped'):
            print(f"  - {job['name']}: SKIPPED (already complete)")
            continue
        time_str = f"{job['time']:.1f}s" if job['time'] < 60 else f"{job['time']/60:.1f}m"
        gpu_str = f" | Peak GPU: {job['gpu_peak']:.2f} GB" if job.get('gpu_peak') else ""
        structs = job.get('structures', '?')
        scores = job.get('scores', 0)
        print(f"  - {job['name']}: {structs} structures, {scores} score files ({time_str}{gpu_str})")
        if job.get('url'):
            print(f"    {job['url']}")

if failed_jobs:
    print(f"\nFailed jobs:")
    for job in failed_jobs:
        error = job.get('error', 'Unknown error')[:80]
        time_str = f"{job['time']:.1f}s" if job.get('time', 0) < 60 else f"{job.get('time', 0)/60:.1f}m"
        print(f"  - {job['name']}: {error} ({time_str})")

# GPU statistics
if gpu_available and completed_jobs:
    gpu_peaks = [j['gpu_peak'] for j in completed_jobs if j.get('gpu_peak') and j['gpu_peak'] > 0]
    if gpu_peaks:
        print(f"\nGPU Peak Statistics:")
        print(f"   Average: {sum(gpu_peaks)/len(gpu_peaks):.2f} GB")
        print(f"   Highest: {max(gpu_peaks):.2f} GB")
        print(f"   Lowest: {min(gpu_peaks):.2f} GB")
        print(f"   Capacity: {total_gpu_vram:.1f} GB")
        print(f"   Peak utilization: {(max(gpu_peaks)/total_gpu_vram)*100:.1f}%")


# ipSAE Analysis Summary
if config.get('enable_ipsae'):
    ipsae_count = sum(1 for j in completed_jobs if j.get('ipsae_ran'))
    print(f"\nipSAE Interface Analysis:")
    print(f"   Per-job: {ipsae_count}/{len(completed_jobs)} jobs analyzed")
    if ipsae_batch_url:
        print(f"   Batch comparison: {ipsae_batch_url}")

if completed_jobs:
    real_jobs = [j for j in completed_jobs if not j.get('skipped') and j.get('time', 0) > 0]
    if real_jobs:
        avg_time = sum(j['time'] for j in real_jobs) / len(real_jobs)
        print(f"\nAverage time per job: {avg_time:.1f}s ({avg_time/60:.1f} min)")

if drive and gdrive_folder_id:
    print(f"\nAll results uploaded to: {gdrive_folder_name}")
    print(f"   https://drive.google.com/drive/folders/{gdrive_folder_id}")
else:
    print(f"\nAll results saved locally in job directories")

print("\nAll done!")
